<a href="https://colab.research.google.com/github/LubnaM/URL-STFN_ExplainPoC/blob/main/URL_STFN_Enroll_HDl_AIiH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Enroll-HD: Disease Stage Discovery via Direct Feature Clustering
# - Same-age cohort selection with max N patients
# - Feature filtering by missingness threshold (20/30/40%)
# - ML-based imputation (IterativeImputer + RandomForest)
# - Grid search for optimal K in KMeans
# ============================================================
'Best than Saliency Map: Guided Backpropagation!! To Test it too!'
'''
SHAP (SHapley Additive exPlanations) model-agnostic method used to explain the output of any machine learning model.
It breaks down a prediction to show the contribution of each feature,
enabling both local (individual prediction) and global (overall model behavior) interpretability.
'''

import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# ------------------------------------------------------------
# 1. Read Enroll-HD data
# ------------------------------------------------------------
df_enroll = pd.read_csv(
    "/content/drive/MyDrive/CHDI/Enroll-HD/enroll.csv", delimiter="\t"
)
df_part = pd.read_csv(
    "/content/drive/MyDrive/CHDI/Enroll-HD/participation.csv", delimiter="\t"
)


# ------------------------------------------------------------
# 2. Keep only gene-positive HD participants (premanifest + manifest)
# ------------------------------------------------------------
hd_subjects = df_part.loc[df_part["hdcat_0"].isin([2, 3]), "subjid"].unique()
df_enroll = df_enroll[df_enroll["subjid"].isin(hd_subjects)].copy()

print("Unique HD participants in manifest and premanifest phases:", df_enroll["subjid"].nunique())




# ------------------------------------------------------------
# 3. Variable selection
# ------------------------------------------------------------
variables = [
    "subjid", "age",

    # Motor
    "ocularh","ocularv","sacinith","sacinitv","sacvelh","sacvelv",
    "dysarth","tongue","fingtapr","fingtapl","prosupr","prosupl",
    "luria","rigarmr","rigarml","brady",
    "dysttrnk","dystrue","dystlue","dystrle","dystlle",
    "chorface","chorbol","chortrnk","chorrue","chorlue","chorrle","chorlle",
    "gait","tandem","retropls",

    # Functional
    "indepscl","carelevl","adl","chores","finances","occupatn",
    #indepscl unique values reach to 100.
    # Cognitive
    "mmsetotal","sit1","verflt05","swrt1","scnt1","verfct5","sdmt1",

    # Post-hoc validation only (NOT clustering)
    "capscore","tfcscore","motscore","diagconf",'hdcat','hdiss_stage_imp'
]

df = df_enroll[variables].copy()

# ------------------------------------------------------------
# 4. Remove visits with all clinical features missing
# ------------------------------------------------------------
clinical_features = [
    c for c in df.columns
    if c not in ["subjid","age","capscore","tfcscore","motscore","diagconf",'hdcat','hdiss_stage_imp']
]

df = df.dropna(subset=clinical_features, how="all")
print("Unique HD participants in manifest and premanifest phases - after clinical visits with all null features:", df["subjid"].nunique())

'''
# ------------------------------------------------------------
# 5. AGE HANDLING: Find age that maximizes number of patients
#    (same age, multiple visits allowed)
#    Selecting the age group with the highest patient
#    representation and restricting the analysis to that cohort.
# ------------------------------------------------------------
n = 1  # number of age cohorts you want

age_patient_counts = (
    df.groupby("age")["subjid"]
    .nunique()
    .sort_values(ascending=False)
)

selected_ages = age_patient_counts.head(n).index

df_n_ages = df[df["age"].isin(selected_ages)].copy()

print(f"Selected age cohorts: {list(selected_ages)}")
print("Patients:", df_n_ages["subjid"].nunique())
print("Visits:", df_n_ages.shape[0])

print("\nPatients per age:")
print(age_patient_counts.loc[selected_ages])
'''
#------------------------------------------------------------
#-- Drop records with age or cap as null/zero
#------------------------------------------------ -----------
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["capscore"] = pd.to_numeric(df["capscore"], errors="coerce")
df = df.dropna(subset=["age", "capscore"])
df = df[(df["age"] > 0) & (df["capscore"] > 0) & (df["hdcat"] < 4)].copy()

print("Unique HD participants in manifest and premanifest phases - after removing those with zero and capscore null or zeros:", df["subjid"].nunique())


#--------------------------------------------
# Exclude patient ID in combined morphometry - keep it for validation
#-------------------------------------------

# 1. Read Combined-Morphometry data
# ------------------------------------------------------------
print(">>Before overlap elimination with morphometrics", df.shape)
print(f"Selected patients: {df['subjid'].unique()}")

df_morph = pd.read_csv(
    "/content/drive/MyDrive/CHDI/SUBJECT_INFO.tsv", delimiter="\t", on_bad_lines='skip')

print(df_morph)
# Get unique Enroll-HD subject IDs
enroll_ids = set(df["subjid"].unique())
# Filter combined_morphometry to only those in Enroll-HD
df_morph_enroll = df_morph[df_morph["sub"].isin(enroll_ids)].copy()
# Get the actual overlapping IDs
overlap_ids = df_morph_enroll["sub"].unique()
# Filter df to exclude overlapping subjects
df = df[~df["subjid"].isin(overlap_ids)].copy()

print("Remaining subjects:", df["subjid"].nunique())
print("Number of overlapping subjects:", overlap_ids)
print(">>After overlap elimination with morphometrics", df.shape)


print("Unique HD participants in manifest and premanifest phases - after excluding validation IDs:", df["subjid"].nunique())


# ------------------------------------------------------------
# AGE HANDLING (YOUR METHOD)
# Select patients with visits at the same ages,
# iteratively maximizing patient overlap
# ------------------------------------------------------------

# Function to perform the iterative age selection, excluding previously selected most common age
def find_most_common_age_n_times(df, n):
    result = []  # To store the results for each iteration
    remaining_patients = df  # Start with all patients
    selected_ages = set()  # Set to store previously selected ages

    for i in range(n):
        # Step 1: Find the most common age in the current remaining patients (excluding selected ages)
        age_counts = remaining_patients['age'].value_counts()
        # Exclude previously selected ages
        age_counts = age_counts[~age_counts.index.isin(selected_ages)]

        if age_counts.empty:
            break  # If there are no more ages to process, break out of the loop

        # Get the most common age
        most_common_age = age_counts.idxmax()
        most_common_age_count = age_counts.max()

        # Step 2: Find patients that have the most common age
        common_age_patients = remaining_patients[remaining_patients['age'] == most_common_age]['subjid'].unique()

        # Add the result for this iteration
        result.append({
            'Iteration': i + 1,
            'Most_Common_Age': most_common_age,
            'Count': most_common_age_count,
            'Patient_IDs': common_age_patients
        })

        # Step 3: Exclude the most common age from the selected ages set
        selected_ages.add(most_common_age)

        # Step 4: Filter the dataset to only keep the patients with the most common age
        remaining_patients = remaining_patients[remaining_patients['subjid'].isin(common_age_patients)]

    return result

# Specify how many iterations you want
n = 4

# Run the function
common_age_results = find_most_common_age_n_times(df, n)
#Get patients from the last iteration
patients = common_age_results[-1]['Patient_IDs']
#Filter df to only these patients
df1 = df[df['subjid'].isin(patients)].copy()
#Find ages that are common to all selected patients
# Count how many unique patients exist for each age
age_counts = df1.groupby('age')['subjid'].nunique()
# Keep only ages where all patients have a visit
common_ages = age_counts[age_counts == len(patients)].index
#Filter df1 to only keep rows within these common ages
df1 = df1[df1['age'].isin(common_ages)].copy() #------------------------------------------->>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
#Remove duplicates if necessary
df1 = df1.drop_duplicates(subset=['subjid', 'age'], keep='last')
print(f"Selected patients: {df1['subjid'].unique()}")
print(f"Common age range: {sorted(common_ages)}")

num_unique_subjects = df1['subjid'].nunique()
print("Number of unique subjects for ", n , "patients: ", num_unique_subjects)

#df1=df #------------------------->>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# ------------------------------------------------------------
# 6. Derive CAG repeats (validation only)
# ------------------------------------------------------------
'Add CAG repeats column'
# Constants for the calculation
L = 30  # Constant L
K = 6.49  # Constant K

# Use .loc to safely assign columns
df1.loc[:, 'age'] = pd.to_numeric(df1['age'], errors='coerce')
df1.loc[:, 'capscore'] = pd.to_numeric(df1['capscore'], errors='coerce')

# Compute CAG safely
df1.loc[:, 'cag'] = ((df1['capscore'] * K) / df1['age'] + L)

# Apply rounding logic
df1.loc[:, 'cag'] = df1['cag'].apply(lambda x: np.floor(x) if x % 1 < 0.5 else np.ceil(x)).astype(int)


#----------------------------------------------
#Exclude those who had medication
#-----------------------------------------------


#have medication
pharmacotx_df = pd.read_csv(
    "/content/drive/MyDrive/CHDI/pharmacotx.csv", sep=r'\t', quotechar='"'
)

# 1️⃣ Clean column names
pharmacotx_df.columns = pharmacotx_df.columns.str.strip().str.replace('"', '')
# 2️⃣ Clean all string values in the DataFrame
# Ensure you are working on a copy
# Clean all string values in a vectorized way
pharmacotx_df = pharmacotx_df.apply(
    lambda col: col.str.strip().str.replace('"', '', regex=False) if col.dtype == "object" else col
)

med_ids = pharmacotx_df["subjid"].unique()
df1 = df1[~df1["subjid"].isin(med_ids)].copy()




# ---------------------------------------
# Replace values > 200 with NaN
# ---------------------------------------
THRESHOLD = 200

features_to_check = [
    c for c in df1.columns
    if c not in ["subjid", "age", "capscore", "tfcscore",
                 "motscore", "diagconf", "cag", "hdcat",'hdiss_stage_imp']
]

df1[features_to_check] = df1[features_to_check].where(
    df1[features_to_check] <= THRESHOLD,
    np.nan
)

# ------------------------------------------------------------
# 7. Prepare clustering features (exclude IDs & derived variables)
# ------------------------------------------------------------
excluded = ["subjid","age","capscore","tfcscore","motscore","diagconf","cag",'hdcat','hdiss_stage_imp']

features_df = df1.drop(columns=excluded)
df_meta=df1.copy(deep=True)


#Drop dubs if exist


# 9. INSPECT MISSINGNESS PER FEATURE
# ------------------------------------------------------------
missing_percent = features_df.isna().mean().sort_values(ascending=False) * 100

print("\nMissingness per feature (%):")
print(missing_percent.round(2))

# ------------------------------------------------------------
# 10. PIPELINE: NaN \u2192 -1 + KMeans++
# ------------------------------------------------------------
import pandas as pd
import numpy as np

from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# ------------------------------------------------------------
# 1. Inspect missingness per feature
# ------------------------------------------------------------
missing_percent = features_df.isna().mean() * 100
missing_summary = missing_percent.sort_values(ascending=False)

print("\nMissingness per feature (%):")
print(missing_summary)
print(len(missing_summary))


# ------------------------------------------------------------
# 2. OPTIONAL: drop features with very high missingness
# ------------------------------------------------------------
threshold = 45  # percent
kept_features = missing_summary[missing_summary < threshold].index
X = features_df[kept_features].copy()

print(f"\nFeatures retained (<{threshold}% missing): {len(kept_features)}")

# ------------------------------------------------------------
# 3. Replace NaN with -1
# ------------------------------------------------------------
print(X)
X_filled = X.fillna(-1)



In [ ]:
# ------------------------------------------------------------
# Check for negative values after cleaning and type conversion
# ------------------------------------------------------------

# Select only the features used for clustering
cluster_features = kept_features

# Exclude nulls and convert to float
df_clean = df1[cluster_features].dropna().astype(float)

# Boolean mask: True if negative value exists in that column
negatives_mask = (df_clean < 0).any()

# List features with negative values
features_with_negatives = negatives_mask[negatives_mask].index.tolist()

if len(features_with_negatives) == 0:
    print("No negative values found in the clustering features.")
else:
    print(f"Features with negative values: {features_with_negatives}")

# Count negative values per feature
negatives_count = (df_clean < 0).sum()
print("\nNumber of negative values per feature (only > 0 shown):")
print(negatives_count[negatives_count > 0])

In [ ]:
'Dataset Charachteristics'
import pandas as pd
# Example dataframe df1 columns: 'subjid', 'age', 'sex', 'cag', 'dcl'

df = df1[df1["age"].isin(common_ages)].copy()


# 1️⃣ Number of unique participants
n_participants = df['subjid'].nunique()

# 2️⃣ Number of observations (rows)
n_observations = df.shape[0]

# 3️⃣ Age: mean ± SD
age_min = df["age"].min()#df1['age'].mean()
age_max = df["age"].max()#df1['age'].std()


'''
# 4️⃣ Sex: female percentage
female_count = (df1['sex'] == 'F').sum()
female_pct = female_count / n_observations * 100
'''

# 5️⃣ CAG repeats: mean ± SD
cag_mean = df['cag'].mean()
cag_std = df['cag'].std()

# 6️⃣ Disease stage by dcl
premanifest_count = (df['diagconf'] < 4).sum()
premanifest_pct = premanifest_count / n_observations * 100

manifest_count = (df['diagconf'] == 4).sum()
manifest_pct = manifest_count / n_observations * 100

# 7️⃣ Create summary dataframe
summary_df = pd.DataFrame({
    'Characteristic': [
        'Number of participants',
        'Number of observations',
        'Age (mean ± SD)',
        #'Female (%)',
        'CAG repeats (mean ± SD)',
        'Premanifest (count, %)',
        'Manifest (count, %)'
    ],
    'Value': [
        n_participants,
        n_observations,
        f"{age_min:.1f} - {age_max:.1f}",
        #f"{female_pct:.1f}%",
        f"{cag_mean:.1f} ± {cag_std:.1f}",
        f"{premanifest_count} ({premanifest_pct:.1f}%)",
        f"{manifest_count} ({manifest_pct:.1f}%)"
    ]
})

print(summary_df)


            Characteristic        Value
0   Number of participants          302
1   Number of observations         1208
2          Age (mean ± SD)  54.0 - 57.0
3  CAG repeats (mean ± SD)   42.5 ± 1.9
4   Premanifest (count, %)  234 (19.4%)
5      Manifest (count, %)  972 (80.5%)


In [ ]:
# ---------------------------------------
# Parameters
# ---------------------------------------
features_to_check = [
    c for c in df1.columns
    if c not in ["subjid", "age", "capscore", "tfcscore",
                 "motscore", "diagconf", "cag", "hdcat"]
]

# ---------------------------------------
# Print unique values per feature
# ---------------------------------------
for feature in features_to_check:
    unique_vals = sorted(df1[feature].dropna().unique())
    print(f"{feature} ({len(unique_vals)} unique values):")
    print(unique_vals)
    print("-" * 50)


ocularh (5 unique values):
[np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
--------------------------------------------------
ocularv (5 unique values):
[np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
--------------------------------------------------
sacinith (5 unique values):
[np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
--------------------------------------------------
sacinitv (5 unique values):
[np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
--------------------------------------------------
sacvelh (5 unique values):
[np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
--------------------------------------------------
sacvelv (5 unique values):
[np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
--------------------------------------------------
dysarth (5 uni

In [ ]:
' ------- Graph-based models : 1) calculate similarity thresholds'

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
print(common_ages)


print(kept_features)
feature_columns = kept_features.copy()

# Step 3: Calculate the 90th percentile similarity per age, then average these thresholds
thresholds_per_age = []
print(df1['age'].unique())
for age in sorted(common_ages):#df1['age'].unique()):
    year_data = df_meta[df_meta['age'] == age].reset_index(drop=True)
    num_patients = len(year_data)

    if num_patients < 2:
        continue  # skip ages with less than 2 patients

    year_data=year_data.fillna(-1)
    similarities = []
    for i in range(num_patients):
        for j in range(i + 1, num_patients):
            patient_i = year_data.loc[i, feature_columns].values.reshape(1, -1)
            patient_j = year_data.loc[j, feature_columns].values.reshape(1, -1)
            sim = cosine_similarity(patient_i, patient_j)[0][0]
            #sim = np.linalg.norm(patient_i - patient_j)  # Euclidean distance
            similarities.append(sim)

    if similarities:
        # Get 95th percentile for this age
        threshold_90 = np.percentile(similarities, 95) #95
        thresholds_per_age.append(threshold_90)

# Calculate mean of 90th percentile thresholds across all ages
average_similar_pairs = np.mean(thresholds_per_age)
print(f"Global similarity threshold (mean of 95th percentiles): {average_similar_pairs:.4f}")


Index([54.0, 55.0, 56.0, 57.0], dtype='float64', name='age')
Index(['mmsetotal', 'verflt05', 'sit1', 'sdmt1', 'swrt1', 'scnt1', 'verfct5',
       'rigarmr', 'retropls', 'rigarml', 'indepscl', 'sacinitv', 'chorlle',
       'chorrle', 'chorlue', 'prosupl', 'fingtapr', 'tongue', 'fingtapl',
       'prosupr', 'sacvelh', 'sacvelv', 'sacinith', 'luria', 'dysttrnk',
       'dystrue', 'dystlue', 'brady', 'dystrle', 'chores', 'finances',
       'chorrue', 'chortrnk', 'chorbol', 'chorface', 'dystlle', 'gait',
       'tandem', 'adl', 'carelevl', 'occupatn', 'dysarth', 'ocularv',
       'ocularh'],
      dtype='object')
[54. 55. 56. 57.]
Global similarity threshold (mean of 95th percentiles): 0.9868


In [ ]:
'FIRST: Visualize Just present the dataset features and its pattern by time'

import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
import matplotlib.cm as cm
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy.stats import pearsonr
from matplotlib.patches import Ellipse  # Import Ellipse for drawing circles
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

threshold=average_similar_pairs

def euclidean_perm_test(vec1, vec2):
    observed = euclidean_distances(vec1.reshape(1, -1), vec2.reshape(1, -1))[0][0]
    return observed#, p_value

def cosine_similarity_score(vec1, vec2):
    # Reshape for pairwise computation
    vec1 = vec1.reshape(1, -1)
    vec2 = vec2.reshape(1, -1)

    # Compute cosine similarity
    similarity = cosine_similarity(vec1, vec2)[0][0]
    return similarity
def adjust_color_intensity(color, intensity):
    """Adjusts the intensity of a color, making it lighter with higher intensity.
    Args:
        color: A tuple or list representing the color (e.g., (R, G, B, A)).
        intensity: A float between 0 and 1 representing the desired intensity.
    Returns:
        A tuple or list representing the adjusted color.
    """
    # Convert color to RGB if it's in other formats

    r, g, b, a = color

    # Calculate how much to lighten each color component
    intensity = np.clip(intensity, 0.1, 1)
    lighten_factor = (1 - intensity)**100
    # Apply the lightening factor
    r = r + (1 - r) * lighten_factor
    g = g + (1 - g) * lighten_factor
    b = b + (1 - b) * lighten_factor

    return (r, g, b, a)  # Return as RGBA tuple



# Unique ages in the dataset (assuming they are integers)
'''
# Filter ages based on starting year and time step
selected_ages = set(df_selected['visit_month'])

feature_columns = df_selected.columns.difference(['patient_id', 'visit_month'])  # Exclude non-feature columns  #^^^^^ To have only motor symptoms

scaler = StandardScaler()
df_selected[feature_columns] = scaler.fit_transform(df_selected[feature_columns])
'''
time_step = 1
# Create subplots
selected_ages = sorted(common_ages.copy())
#selected_ages = sorted(df1['age'].unique()) #####common_ages

num_plots = len(selected_ages)
num_cols = min(num_plots, 3)  # Limit columns to 4
num_rows = (num_plots + num_cols - 1) // num_cols  # Calculate rows needed

fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols * 7, num_rows * 7))
fig.subplots_adjust(left=0.1, right=1, bottom=0.1, top=0.9)

graphs=[]
# Iterate through selected ages and create graphs
for plot_index, age in enumerate(selected_ages): # Change i to plot_index
    year_data = df_meta[df_meta['age'] == age]
    year_data=year_data.fillna(-1)
    G = nx.Graph()
    added_edges = set()
    edge_thickness = {}

    # Add nodes for this age group
    for _, row in year_data.iterrows():
        # Add original features directly to the node
        features = {feature: row[feature] for feature in feature_columns}
        G.add_node(row['subjid'], **features)  # ,age=row['age'], removed

    for i, row1 in year_data.iterrows():
        for j, row2 in year_data.iterrows():
            if i != j and row1['subjid'] != row2['subjid']:
                pair = tuple(sorted((row1['subjid'], row2['subjid'])))
                if pair not in added_edges:
                    similarity= cosine_similarity_score(row1[feature_columns].values.astype(float), row2[feature_columns].values.astype(float))
                    if abs(similarity) >= threshold:
                        G.add_edge(row1['subjid'], row2['subjid'], weight=similarity)
                        added_edges.add(pair)

    # ----> Append the graph to the list <----
    graphs.append(G)


   # Plotting on the current subplot
    pos = nx.spring_layout(G, seed=15, k=0.4, scale=1) # Initial layout
    # 2. Distinct node color for each 'subjid' with fade-out
    unique_subjids = list(G.nodes())
    num_subjids = len(unique_subjids)
    color_map_subjid = plt.get_cmap('tab20', num_subjids)
    node_colors_subjid = [
        adjust_color_intensity(color_map_subjid(unique_subjids.index(n)),
                                np.mean([G.nodes[n].get(feature, 0) for feature in feature_columns]))
        for n in G.nodes()
    ]


    # Draw nodes once outside the loop:
    # Draw nodes once outside the loop:
    row_index = plot_index // num_cols
    col_index = plot_index % num_cols

    if num_plots > 1:
        ax = axes[row_index, col_index] if num_rows > 1 else axes[col_index]
    else:
        ax = axes

    nx.draw_networkx_nodes(G, pos, node_color=node_colors_subjid,node_size=2000, linewidths=2, alpha=0.9, ax=ax)
    # Draw edges with boundary and color based on CAG
    # Draw edges with correlation-based color only
    nx.draw_networkx_edges(G, pos, width=[edge_thickness.get(edge, 1) * 2 for edge in G.edges()],
                               edge_color='Red', alpha=0.8, ax=ax)#axes[plot_index])


    # Add labels:
    labels = {node: node for node in G.nodes()}
    # Pass the correct Axes object using row_index and col_index:
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=8, font_color='black', ax=axes[row_index, col_index] if num_rows > 1 else axes[col_index])
    # Set title for the current subplot correctly:
    (axes[row_index, col_index] if num_rows > 1 else axes[col_index]).set_title(f"Age {age}")
# Adjust layout and display the plot
#fig.set_size_inches(15, 8)  # Adjust overall figure size

plt.tight_layout()
plt.show()

'''
X and Y Coordinates: Primarily determine by the spring layout algorithm to reduce visual crossing of edges.
Feature Similarity: Nodes with similar patterns of feature values are more likely to be positioned near each other.
Interconnectedness: Nodes connected by stronger (high correlation coefficient value) are often positioned closer to each other.
'''

In [ ]:
'STRUCTURED'
'MultiModel (1/2) - Model Training- original'
!pip install torch_geometric
#pip install tensorboard
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from sklearn.preprocessing import (
    StandardScaler, RobustScaler, LabelEncoder, normalize
)
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score
)
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
#=============== fix random weight initialization ===========#

import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

import random
# This ensures fixed weight initialization for the neural network
seeds_c=10

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(seeds_c)

def shoulson_stage(tfc):
    if pd.isna(tfc):
        return np.nan
    if tfc >= 11:
        return 1
    elif tfc >= 7:
        return 2
    elif tfc >= 3:
        return 3
    elif tfc >= 1:
        return 4
    elif tfc == 0:
        return 5
    else:
        return np.nan
# ----------------------------
# 2. Global node mapping
# ----------------------------
node_id_maps = [list(g.nodes) for g in graphs]
age_list = selected_ages#.tolist()  #------------------------------->>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

all_nodes_global = sorted(set(n for nodes in node_id_maps for n in nodes))
global_node_to_index = {n: i for i, n in enumerate(all_nodes_global)}

# T, N, F_dim need to be defined based on these
T_timesteps = len(graphs) # Number of time steps
N_subjects_global = len(all_nodes_global) # Use len(all_nodes_global) instead of num_nodes_global to ensure it's defined

# Moved F_dim_features definition here
F_dim_features = len(feature_columns) # Feature dimension


# ----------------------------
# 3. Global feature scaling
# ----------------------------

# Fit ONCE on all data
all_features = []
for graph in graphs:
    for node in graph.nodes:
        all_features.append([graph.nodes[node][f] for f in feature_columns])

global_scaler = StandardScaler()
global_scaler.fit(np.array(all_features))


# ----------------------------
# 4. Build PyG data (GLOBAL order)
# ----------------------------
all_data = []
for g in graphs:
    x = np.zeros((N_subjects_global, F_dim_features))
    for n in g.nodes:
        x[global_node_to_index[n]] = [g.nodes[n][f] for f in feature_columns]
    x = global_scaler.transform(x)

    if g.edges:
        edge_index = np.array(
            [(global_node_to_index[u], global_node_to_index[v]) for u, v in g.edges]
        ).T
    else:
        edge_index = np.empty((2, 0), dtype=int)

    all_data.append(
        Data(
            x=torch.tensor(x, dtype=torch.float),
            edge_index=torch.tensor(edge_index, dtype=torch.long)
        )
    )

# ----------------------------
# 5. id_df (GLOBAL order)
# ----------------------------
records = []
idx = 0
for t in range(T_timesteps):
    for node in all_nodes_global:
        records.append({
            "row_id": idx,
            "subjid": node,
            "age": age_list[t]
        })
        idx += 1

id_df = pd.DataFrame(records)

# ----------------------------
# 7. Loss
# ----------------------------
#edge reconstruction

def reconstruction_loss(embeddings, edge_index_seq, mask):
    loss = 0
    T, N, _ = embeddings.shape
    for t in range(T):
        emb = embeddings[t]
        pred = torch.sigmoid(emb @ emb.T)
        target = torch.zeros((N, N), device=emb.device)
        ei = edge_index_seq[t]
        target[ei[0], ei[1]] = 1.0
        node_mask = mask[t].unsqueeze(1) & mask[t].unsqueeze(0)
        if node_mask.sum() > 0:
            loss += F.binary_cross_entropy(pred[node_mask], target[node_mask])
    return loss




# URL-STFN
'''
Thie method will:

1) Spatial encoding (GCN per timestep)
Apply the same GCNConv to each snapshot
Output shape: [T, N, gcn_out]

2)Temporal encoding (GRU across time, per node)
Permute to [N, T, gcn_out]
Run a single-layer GRU
Take the final hidden state → one temporal embedding per node
Output shape: [N, gru_hidden]

3)Broadcast temporal features across time
Expand to [T, N, gru_hidden]
Fuse spatial + temporal
Concatenate → [T, N, gcn_out + gru_hidden]

4) Pass through the same MLP fusion head
Outputs are identical in shape and meaning:
[T, N, fuse]
'''


class SpatioTemporalEncoder(nn.Module):
    def __init__(self, input_dim, gcn_out, gru_hidden, fuse):
        super().__init__()
        self.gcn = GCNConv(input_dim, gcn_out)
        self.gru = nn.GRU(gcn_out, gru_hidden, batch_first=True)
        self.fusion = nn.Sequential(
            nn.Linear(gcn_out + gru_hidden, fuse),
            nn.ReLU()#,
            #nn.LayerNorm(fuse)
        )
        self.gcn_decoder = GCNConv(fuse, input_dim)


    def forward(self, x_seq, edge_index_seq):
        T = x_seq.size(0)
        spatial = torch.stack([
            self.gcn(x_seq[t], edge_index_seq[t]) for t in range(T)
        ])
        temporal_input = spatial.permute(1, 0, 2)
        _, h = self.gru(temporal_input)
        temporal = h.squeeze(0).unsqueeze(0).expand(T, -1, -1)
        combined = torch.cat([spatial, temporal], dim=-1)
        fused = self.fusion(combined)  # ✅ define fused

        # ---------- Decode ----------
        reconstructed = torch.stack([
            self.gcn_decoder(fused[t], edge_index_seq[t]) for t in range(T)
        ])  # [T, N, input_dim]

        return fused, reconstructed


# ----------------------------
# 8. Train & embed
# ----------------------------
def train_and_embed(i, j, k, x_seq, edge_index_seq, mask, seed):
    set_seed(seed)
    model = SpatioTemporalEncoder(F_dim_features, i, j, k)
    opt = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_history = []   #


    model.train()
    for _ in range(500):
        opt.zero_grad()
        #emb = model(x_seq, edge_index_seq)
        #loss = reconstruction_loss(emb, edge_index_seq, mask)
        fused_embedding, reconstructed = model(x_seq, edge_index_seq)
        #loss = F.mse_loss(reconstructed[mask], x_seq[mask])

        # Node feature reconstruction (masked)

        loss = F.mse_loss(
            reconstructed[mask],
            x_seq[mask],
            reduction="sum"
        ) / mask.sum()

        loss.backward()
        opt.step()
        loss_history.append(loss.item())  #


    model.eval()
    with torch.no_grad():
      fused_embedding, _ = model(x_seq, edge_index_seq)
    return fused_embedding.cpu().numpy(), loss_history, model


# ----------------------------
# 9. Clustering
# ----------------------------
def cluster_embeddings(embeddings, seed, n_clusters=2):
    T, N, D = embeddings.shape
    flat = embeddings.reshape(T * N, D)
    flat = StandardScaler().fit_transform(flat)
    #flat = normalize(flat, norm="l2")
    km = KMeans(n_clusters=n_clusters,init='k-means++', n_init=10, random_state=seed)
    return km.fit_predict(flat)

# ----------------------------
# 10. ARI evaluation
# ----------------------------
def compute_ari(cluster_labels, id_df, df_meta):
    tmp = id_df.copy()
    tmp["cluster"] = cluster_labels

    merged = df_meta.merge(tmp, on=["subjid", "age"], how="inner")

    # -----------------------------
    # 1️⃣ HD category ARI
    # -----------------------------
    merged_hdcat = merged.dropna(subset=["hdcat"]).copy()
    # ❗ No tfc filtering here → hdcat always evaluated on full data

    if merged_hdcat["hdcat"].nunique() < 2:
        ari_hdcat = np.nan
    else:
        le = LabelEncoder()
        hdcat_labels = le.fit_transform(merged_hdcat["hdcat"])
        ari_hdcat = adjusted_rand_score(
            hdcat_labels,
            merged_hdcat["cluster"]
        )

    # -----------------------------
    # 2️⃣ Shoulson ARI (TFC-based)
    # -----------------------------
    merged_shoulson = merged.copy()

    # 🚨 THIS is where tfcscore filtering happens
    merged_shoulson = merged_shoulson[
        merged_shoulson["tfcscore"].notna() &
        (merged_shoulson["tfcscore"] != -1)
    ].copy()

    merged_shoulson["shoulson_stage"] = merged_shoulson["tfcscore"].apply(shoulson_stage)
    merged_shoulson = merged_shoulson.dropna(subset=["shoulson_stage"])

    if merged_shoulson["shoulson_stage"].nunique() < 2:
        ari_shoulson = np.nan
    else:
        ari_shoulson = adjusted_rand_score(
            merged_shoulson["shoulson_stage"],
            merged_shoulson["cluster"]
        )

    return ari_hdcat, ari_shoulson



# ----------------------------
# 11. Prepare sequences
# ----------------------------
x_seq = torch.stack([d.x for d in all_data])
edge_index_seq = [d.edge_index for d in all_data]
mask = (x_seq.sum(dim=-1) != 0)

# ----------------------------
# 12. Grid search (BEST ARI)
# ----------------------------
from collections import defaultdict
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.metrics import silhouette_score
import numpy as np
'''
NN = [4, 8, 16, 32, 64]
num_seeds = 10
'''
max_k_limit = 20

results = defaultdict(list)

global_best = {
    "silhouette": -1,
    "i": None,
    "j": None,
    "k": None,
    "best_k": None,
    "seed": None
}

# ============================================================
# GRID SEARCH
# ============================================================
'''
for i in NN:
    for j in NN:
        for k_dim in NN:
            for seed in range(num_seeds):
                set_seed(seed)
'''
i=4
j=4
k_dim=4
seed=seeds_c
# ---- Train & embed (URL-STFN encoder) ----
embeddings, loss_history,model = train_and_embed(
    i, j, k_dim,
    x_seq,
    edge_index_seq,
    mask,
    seed
)  # [T, N, k_dim]

'plot the loss'
plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss Curve")
plt.grid(True)
plt.show()
# ---- Flatten [T, N, D] → [T*N, D] ----
T, N, D = embeddings.shape
emb_2d = embeddings.reshape(T * N, D)

# ---- Normalize ----
#emb_2d = StandardScaler().fit_transform(emb_2d)
#emb_2d = normalize(emb_2d, norm="l2")
'''
# ---- Silhouette grid search ----
best_sil = -1
best_k = None
best_labels = None
#It’s a rule-of-thumb often used in clustering: you shouldn’t try more clusters than roughly the square root of your dataset size.
k_max = min(int(np.sqrt(len(emb_2d))), max_k_limit)
for kk in range(2, k_max + 2):
    labels = KMeans(
        n_clusters=kk,
        init="k-means++",
        n_init=10,
        random_state=seed
    ).fit_predict(emb_2d)

    if len(np.unique(labels)) < 2:
        continue

    sil = silhouette_score(emb_2d, labels)
    if sil > best_sil:
        best_sil = sil
        best_k = kk
        best_labels = labels
        dbi=davies_bouldin_score(emb_2d, labels)
        chz=calinski_harabasz_score(emb_2d, labels)
        # ---- External evaluation ----
        ari_hd, ari_sh = compute_ari(
            best_labels,
            id_df.copy(),
            df_meta
        )
'''

'''
                # ---- External evaluation ----
                ari_hd, ari_sh = compute_ari(
                    best_labels,
                    id_df.copy(),
                    df_meta
                )

                results[(i, j, k_dim)].append({
                    "seed": seed,
                    "silhouette": best_sil,
                    "db":dbi,
                    "ch":chz,
                    "best_k": best_k,
                    "ari_hdcat": ari_hd,
                    "ari_shoulson": ari_sh
                })

                # ---- GLOBAL BEST ----
                if best_sil > global_best["silhouette"]:
                    global_best.update({
                        "silhouette": best_sil,
                        "db":dbi,
                        "ch":chz,
                        "i": i,
                        "j": j,
                        "k": k_dim,
                        "best_k": best_k,
                        "seed": seed
                    })

            # ====================================================
            # Aggregate per (i, j, k)
            # ====================================================
            runs = results[(i, j, k_dim)]
            sils = [r["silhouette"] for r in runs]
            db   = [r["db"] for r in runs]
            ch   = [r["ch"] for r in runs]
            ks   = [r["best_k"] for r in runs]

            print(
                f"(i={i}, j={j}, k={k_dim}) | "
                f"Mean Sil={np.nanmean(sils):.3f}±{np.nanstd(sils):.3f}, Mean db={np.nanmean(db):.3f}±{np.nanstd(db):.3f}, Mean chz={np.nanmean(ch):.3f}±{np.nanstd(ch):.3f}, "
                f"best k={best_k:.1f}, best Sil={best_sil:.1f} "
                f"ARI_hdcat={np.nanmean([r['ari_hdcat'] for r in runs]):.3f} ± {np.nanstd([r['ari_hdcat'] for r in runs]):.3f}, "
                f"ARI_shoulson={np.nanmean([r['ari_shoulson'] for r in runs]):.3f} ± {np.nanstd([r['ari_shoulson'] for r in runs]):.3f}"
            )

print("\n🏆 GLOBAL BEST CONFIGURATION")
print(
    f"Best silhouette = {global_best['silhouette']:.4f}\n"
    f"Best db = {global_best['db']:.4f}\n"
    f"Best ch = {global_best['ch']:.4f}\n"
    f"i = {global_best['i']}\n"
    f"j = {global_best['j']}\n"
    f"k_dim = {global_best['k']}\n"
    f"best_k = {global_best['best_k']}\n"
    f"seed = {global_best['seed']}"
)
'''

"""
# ============================ CLUSTER ALL EMBEDDINGS TOGETHER ============================
T, N, D = all_embeddings.shape
flat = all_embeddings.reshape(T * N, D)
flat = StandardScaler().fit_transform(flat)
flat = normalize(flat, norm="l2")

# Grid search to find best k using silhouette score
k_max = min(len(unique_subj_ids), int(math.sqrt(len(unique_subj_ids))), max_limit)
range_n_clusters=list(range(2, k_max + 1))
best_k = 2
best_score = -1
best_labels = None

for k in range_n_clusters:
  km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=seeds_c)
  labels = km.fit_predict(flat)
  score = silhouette_score(flat, labels)
  if score > best_score:
      best_k = k
      best_score = score
      best_labels = labels
"""
best_k=4
# Final clustering using best_k
final_kmeans = KMeans(n_clusters=best_k, init='k-means++', n_init=10, random_state=seeds_c)
cluster_labels = final_kmeans.fit_predict(emb_2d)  # shape: [T*N]


# Compute other metrics for the best clustering ONLY if more than one cluster exists
if len(np.unique(cluster_labels)) > 1:
    db_index = davies_bouldin_score(emb_2d, cluster_labels)
    ch_index = calinski_harabasz_score(emb_2d, cluster_labels)
    best_score = silhouette_score(emb_2d, cluster_labels)

    print(f"Best K: {best_k}")
    print(f"Silhouette Score: {best_score:.4f}")
    print(f"Davies-Bouldin Index: {db_index:.4f}")
    print(f"Calinski-Harabasz Index: {ch_index:.4f}")
else:
    print(f"Clustering resulted in only 1 cluster at best k={{best_k}}. Skipping metrics.")


id_df["cluster"] = cluster_labels
heatmap_df = id_df.pivot(
    index="subjid",
    columns="age",
    values="cluster"
)
cmap=sns.color_palette("crest", as_cmap=True)

# Clustered heatmap (rows = nodes, cols = time steps)
sns.clustermap(
    heatmap_df,
    cmap=cmap,
    linewidths=0.5,
    linecolor='gray',
    row_cluster=True,
    col_cluster=False,
    figsize=(15, 10)
)

plt.suptitle("Hierarchical Clustering of Nodes and Time Steps", y=1.02)
plt.show()




In [ ]:
;;

In [ ]:
# ------------------------------
# 1️⃣ Flatten and normalize embeddings
# ------------------------------
T, N, D = embeddings.shape
emb_2d = embeddings.reshape(T * N, D)
emb_2d = StandardScaler().fit_transform(emb_2d)

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import numpy as np
from itertools import combinations

def jaccard_coefficient(labels_a, labels_b):
    """
    Jaccard similarity based on co-clustered pairs.
    """
    n = len(labels_a)
    agree_same = 0
    union_same = 0

    for i, j in combinations(range(n), 2):
        same_a = labels_a[i] == labels_a[j]
        same_b = labels_b[i] == labels_b[j]
        if same_a or same_b:
            union_same += 1
        if same_a and same_b:
            agree_same += 1

    return agree_same / union_same if union_same > 0 else 0

# ------------------------------
# 2️⃣ Parameters
# ------------------------------
k_max = min(int(np.sqrt(len(emb_2d))), max_k_limit)
n_bootstrap = 10
sample_fraction = 0.80
stability_threshold = 0.8
best_stable_k = 2

rng = np.random.default_rng(seed+50)

# ------------------------------
# 3️⃣ Bootstrap stability evaluation
# ------------------------------
for k in range(2, k_max + 1):
    jaccards = []

    for b in range(n_bootstrap):
        # Bootstrap sample
        idx = rng.choice(
            len(emb_2d),
            size=int(sample_fraction * len(emb_2d)),
            replace=True
        )
        sample = emb_2d[idx]

        # Fit KMeans on bootstrap sample
        km = KMeans(n_clusters=k, n_init=10, random_state=seed)
        labels_boot = km.fit_predict(sample)

        # Fit on full data and restrict to bootstrap indices
        labels_full = KMeans(
            n_clusters=k,
            n_init=10,
            random_state=seed
        ).fit_predict(emb_2d)

        jaccards.append(
            jaccard_coefficient(labels_boot, labels_full[idx])
        )

    mean_stability = np.mean(jaccards)
    print(f"k={k}, bootstrap Jaccard={mean_stability:.3f}")

    if mean_stability < stability_threshold:
        print(f"Stability dropped below {stability_threshold}, stopping at k={k-1}")
        break

    best_stable_k = k

print("Chosen number of clusters:", best_stable_k)


In [ ]:
'''
OT:
It does this by:
Building a contingency matrix(CM)
Turning it into a cost matrix (-CM), and readjust the labels to match each others
Solving a linear assignment optimization problem
comput stability by = number of matching labels / Total samples
----------------------------------------------------------------------------------
The code below:
For each k:
-Cluster full dataset
-For 10 bootstrap samples:
  Recluster
  Align bootstrap labels to full labels using Hungarian
  Measure agreement (after alignment)
-Compute mean stability across 10 runs
-Stop when mean stability < 0.8
So OT ensures:
-You are comparing cluster structure
-Not arbitrary label numbering
-Stability reflects true structural reproducibility
'''
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.optimize import linear_sum_assignment

# ------------------------------
# 1️⃣ Flatten and normalize embeddings
# ------------------------------
T, N, D = embeddings.shape
emb_2d = embeddings.reshape(T * N, D)
emb_2d = StandardScaler().fit_transform(emb_2d)

# ------------------------------
# 2️⃣ Parameters
# ------------------------------
k_max = min(int(np.sqrt(len(emb_2d))), max_k_limit)
n_bootstrap = 10
sample_fraction = 0.80
stability_threshold = 0.8
best_stable_k = 2

rng = np.random.default_rng(seed+50)

# ------------------------------
# 3️⃣ OT Alignment Function
# ------------------------------
def align_labels_ot(labels_ref, labels_target, n_classes):
    """
    Align labels_target to labels_ref using Hungarian algorithm
    """
    contingency = np.zeros((n_classes, n_classes))

    for i in range(n_classes):
        for j in range(n_classes):
            contingency[i, j] = np.sum(
                (labels_ref == i) & (labels_target == j)
            )

    cost_matrix = -contingency  # maximize overlap
    row_ind, col_ind = linear_sum_assignment(cost_matrix)

    label_map = dict(zip(col_ind, row_ind))
    aligned_labels = np.array([label_map[l] for l in labels_target])

    return aligned_labels

# ------------------------------
# 4️⃣ Bootstrap Stability using OT only
# ------------------------------
for k in range(2, k_max + 1):

    ot_scores = []

    # Cluster full dataset once
    km_full = KMeans(n_clusters=k, n_init=10, random_state=seed)
    labels_full = km_full.fit_predict(emb_2d)

    for b in range(n_bootstrap):

        # Bootstrap sampling
        idx = rng.choice(
            len(emb_2d),
            size=int(sample_fraction * len(emb_2d)),
            replace=True
        )

        sample = emb_2d[idx]

        # Cluster bootstrap sample
        km_boot = KMeans(n_clusters=k, n_init=10, random_state=seed)
        labels_boot = km_boot.fit_predict(sample)

        # Align labels using OT
        labels_boot_aligned = align_labels_ot(
            labels_full[idx],
            labels_boot,
            n_classes=k
        )

        # Compute OT-based stability (label agreement)
        agreement = np.mean(labels_boot_aligned == labels_full[idx])
        ot_scores.append(agreement)

    mean_stability = np.mean(ot_scores)

    print(f"k={k}, OT stability={mean_stability:.3f}")

    if mean_stability < stability_threshold:
        print(f"Stability dropped below {stability_threshold}, stopping at k={k-1}")
        break

    best_stable_k = k

print("Chosen number of clusters:", best_stable_k)


In [ ]:
best_k=4
# Final clustering using best_k
final_kmeans = KMeans(n_clusters=best_k, init='k-means++', n_init=10, random_state=seeds_c)
cluster_labels = final_kmeans.fit_predict(emb_2d)  # shape: [T*N]


# Compute other metrics for the best clustering ONLY if more than one cluster exists
if len(np.unique(cluster_labels)) > 1:
    db_index = davies_bouldin_score(emb_2d, cluster_labels)
    ch_index = calinski_harabasz_score(emb_2d, cluster_labels)
    best_score = silhouette_score(emb_2d, cluster_labels)

    print(f"Best K: {best_k}")
    print(f"Silhouette Score: {best_score:.4f}")
    print(f"Davies-Bouldin Index: {db_index:.4f}")
    print(f"Calinski-Harabasz Index: {ch_index:.4f}")
else:
    print(f"Clustering resulted in only 1 cluster at best k={{best_k}}. Skipping metrics.")


id_df["cluster"] = cluster_labels
heatmap_df = id_df.pivot(
    index="subjid",
    columns="age",
    values="cluster"
)
cmap=sns.color_palette("crest", as_cmap=True)

# Clustered heatmap (rows = nodes, cols = time steps)
sns.clustermap(
    heatmap_df,
    cmap=cmap,
    linewidths=0.5,
    linecolor='gray',
    row_cluster=True,
    col_cluster=False,
    figsize=(15, 10)
)

plt.suptitle("Hierarchical Clustering of Nodes and Time Steps", y=1.02)
plt.show()

'''
4, 8, 32  --- n = 5
32, 8, 4  --- n = 6 longitudinal records.
'''

In [ ]:
# ============================================================
# 🔍 Explainability: Feature Importance Over Time
# ============================================================

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import copy
from sklearn.preprocessing import MinMaxScaler

# ============================================================
# 1️⃣ SALIENCY MAP (Over ALL time steps)
# ============================================================

all_saliency = []

for t in range(T):

    # ---- Prepare input with gradients ----
    x_seq_grad = x_seq.clone().detach()

    x_t = x_seq_grad[t].clone().detach().requires_grad_(True)
    x_t.retain_grad()  # 🔥 critical fix

    x_seq_grad[t] = x_t

    # ---- Forward ----
    model.zero_grad()
    fused, _ = model(x_seq_grad, edge_index_seq)

    # ---- Target (embedding magnitude) ----
    target = fused[t].norm(dim=1).mean()

    # ---- Backward ----
    target.backward()

    # ---- Extract saliency ----
    saliency_t = x_t.grad.abs().cpu().numpy()  # [N, F]

    # ---- Aggregate across nodes ----
    saliency_mean = saliency_t.mean(axis=0)  # [F]

    all_saliency.append(saliency_mean)

# ---- Convert to matrix ----
# shape: [T, F] → transpose → [F, T]
saliency_matrix = np.array(all_saliency).T


# ============================================================
# 2️⃣ GUIDED BACKPROPAGATION (Over ALL time steps)
# ============================================================

# ---- Custom Guided ReLU ----
class GuidedReLU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)
        return input.clamp(min=0)

    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        grad_input = grad_output.clone()
        grad_input[input < 0] = 0
        grad_input[grad_output < 0] = 0
        return grad_input


# ---- Replace ReLU safely ----
def replace_relu_with_guided(module):
    for name, mod in module.named_children():
        if isinstance(mod, nn.ReLU):
            module._modules[name] = nn.Sequential()
            module._modules[name].forward = lambda x: GuidedReLU.apply(x)
        else:
            replace_relu_with_guided(mod)


# ---- Clone trained model ----
guided_model = copy.deepcopy(model)
replace_relu_with_guided(guided_model)
guided_model.eval()

guided_all = []

for t in range(T):

    x_seq_grad = x_seq.clone().detach()

    x_t = x_seq_grad[t].clone().detach().requires_grad_(True)
    x_t.retain_grad()

    x_seq_grad[t] = x_t

    # ---- Forward ----
    guided_model.zero_grad()
    fused_g, _ = guided_model(x_seq_grad, edge_index_seq)

    # ---- Target ----
    target_g = fused_g[t].norm(dim=1).mean()

    # ---- Backward ----
    target_g.backward()

    guided_saliency_t = x_t.grad.abs().cpu().numpy()

    guided_all.append(guided_saliency_t.mean(axis=0))

# ---- Convert to matrix ----
guided_matrix = np.array(guided_all).T  # [F, T]


# ============================================================
# 3️⃣ OPTIONAL NORMALIZATION (better visualization)
# ============================================================

scaler = MinMaxScaler()

saliency_matrix_norm = scaler.fit_transform(saliency_matrix)
guided_matrix_norm = scaler.fit_transform(guided_matrix)


# ============================================================
# 4️⃣ VISUALIZATION
# ============================================================

# Saliency with row clustering
# -----------------------------
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(16, 10))
sns.clustermap(
    saliency_matrix_norm,
    cmap="viridis",
    xticklabels=age_list,
    yticklabels=feature_columns,
    row_cluster=True,       # 🔹 hierarchical clustering on features
    col_cluster=False,      # keep age/time in order
    linewidths=0.5,
    linecolor='gray',
    figsize=(14, 10)
)
plt.title("Saliency Map with Feature Clustering", pad=50)
plt.show()


# Guided Backprop with row clustering
# -----------------------------
sns.clustermap(
    guided_matrix_norm,
    cmap="magma",
    xticklabels=age_list,
    yticklabels=feature_columns,
    row_cluster=True,       # 🔹 hierarchical clustering on features
    col_cluster=False,
    linewidths=0.5,
    linecolor='gray',
    figsize=(14, 10)
)
plt.title("Guided Backpropagation with Feature Clustering", pad=50)
plt.show()


# In tabular form
#---------------------------------------
import pandas as pd
import seaborn as sns

# ---- Saliency clustermap ----
cg = sns.clustermap(
    saliency_matrix_norm,
    cmap="viridis",
    xticklabels=age_list,
    yticklabels=feature_columns,
    row_cluster=True,
    col_cluster=False,
    linewidths=0.5,
    linecolor='gray',
    figsize=(14, 10)
)

plt.title("Saliency Map with Feature Clustering", pad=50)
plt.show()

# ---- Extract the order of rows (features) after clustering ----
row_order = cg.dendrogram_row.reordered_ind
clustered_features = [feature_columns[i] for i in row_order]

# ---- Rearrange saliency matrix according to clustered features ----
saliency_clustered = saliency_matrix_norm[row_order, :]

# ---- Convert to DataFrame for tabular display ----
df_saliency_clustered = pd.DataFrame(
    saliency_clustered,
    index=clustered_features,   # Features after clustering
    columns=age_list             # Timepoints
)

# ---- Display the table ----
print("Saliency Values (Features × Time) - Clustered Rows")
display(df_saliency_clustered)

In [ ]:
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

'SHAP Answers: which features caused a patient to be assigned to a stage ❓'
'“i.e. Which features, at which ages, define each discovered disease stage.”'
# ============================================================
# Calculate Labels for specific number of clusters:
#=============================================================
# Final clustering using best_k
k_val=4
final_kmeans = KMeans(n_clusters=k_val, init='k-means++', n_init=10, random_state=seeds_c)
cluster_labels = final_kmeans.fit_predict(emb_2d)  # shape: [T*N]

id_df["cluster"] = cluster_labels
heatmap_df = id_df.pivot(
    index="subjid",
    columns="age",
    values="cluster"
)
# ============================================================
# 1️⃣ Train surrogate classifier (embeddings → clusters)
# ============================================================
clf = RandomForestClassifier(n_estimators=100, random_state=seeds_c)
clf.fit(emb_2d, cluster_labels)

# ============================================================
# 2️⃣ Wrapper
# ============================================================
# Wrapper answers:
# “If I change feature X, how does cluster assignment change?”
def model_wrapper_features(x_numpy, t_idx):
    x_tensor = torch.tensor(x_numpy, dtype=torch.float)

    x_seq_input = x_tensor.unsqueeze(0)
    edge_idx = [edge_index_seq[t_idx]]

    with torch.no_grad():
        fused, _ = model(x_seq_input, edge_idx)

    emb_flat = fused.reshape(x_tensor.shape[0], -1).cpu().numpy()
    return clf.predict_proba(emb_flat)
'''
predict_proba gives SHAP a smooth “scale” to measure feature impact; Returns probabilities for each class (cluster); if they are 4 clusters, predict_proba returns: [[0.1, 0.7, 0.1, 0.1]] : .
predict gives only a label → too discrete → SHAP can’t quantify contributions well.
'''



'''
15-20% Background sample is sourced from https://arxiv.org/abs/1705.07874;
Following SHAP recommendations, we used a subset of 50 patients as the reference background set for computing feature attributions, representing ~15–20% of the cohort, balancing computational efficiency and stable estimates (Lundberg & Lee, 2017).
'''
# ============================================================
# 3️⃣ Compute SHAP once per timestep (shared across clusters)
# ============================================================
shap_values_list = []

for t_idx in range(T):

    X_t = np.array(all_data[t_idx].x)
    '''
    Intuition: SHAP measures how much each feature pushes the model away from the “typical patient” prediction.
    You simply took the 50 random patients at this timestep as the reference population.
    The background in SHAP is your reference population
    It tells SHAP:
    “This is what ‘normal’ or ‘typical’ patients look like.”
    Then, SHAP asks for each patient:
    “How does this patient differ from the typical ones? Which features push the prediction up or down?”
    '''
    # ✅ RANDOM background instead of first 50
    n_bg = min(100, len(X_t))
    idx = np.random.choice(len(X_t), size=n_bg, replace=False)
    background = X_t[idx]

    explainer = shap.Explainer(
        lambda x: model_wrapper_features(x, t_idx),
        background
    )

    shap_values = explainer(X_t)  # [N, F, C]
    shap_values_list.append(shap_values)
    # background-based Shapley calculation (subset sampling), not permutation.

# ============================================================
# 4️⃣ LOOP OVER ALL CLUSTERS
# ============================================================
for cluster_id in range(k_val):

    print(f"\n================ CLUSTER {cluster_id} ================\n")

    shap_time_feature = []

    for t_idx, sv in enumerate(shap_values_list):

        # select cluster
        sv_cluster = sv.values[:, :, cluster_id]  # [N, F]

        # aggregate across nodes
        sv_mean = np.mean(np.abs(sv_cluster), axis=0)  # [F]

        shap_time_feature.append(sv_mean)

    # [T, F] → [F, T]
    shap_matrix = np.array(shap_time_feature).T

    # ========================================================
    # Normalize
    # ========================================================
    scaler = MinMaxScaler()
    shap_matrix_norm = scaler.fit_transform(shap_matrix)

    # ========================================================
    # Create DataFrame (IMPORTANT)
    # ========================================================
    shap_df = pd.DataFrame(
        shap_matrix_norm,
        index=feature_columns,
        columns=age_list
    )

    # ========================================================
    # 5️⃣ Heatmap (row clustered)
    # ========================================================
    sns.set(font_scale=0.9)
    sns.set_style("whitegrid")

    g = sns.clustermap(
        shap_df,
        cmap="RdBu_r",
        row_cluster=True,
        col_cluster=False,
        linewidths=0.5,
        figsize=(14, 8)
    )

    plt.title(f"SHAP Feature Importance Over Time (Cluster {cluster_id})", pad=20)
    plt.xlabel("Age (Time)")
    plt.ylabel("Features")
    plt.show()

    # ========================================================
    # 6️⃣ Extract row-clustered order
    # ========================================================
    row_order = g.dendrogram_row.reordered_ind
    shap_df_clustered = shap_df.iloc[row_order]

    # ========================================================
    # 7️⃣ PRINT TABULAR RESULTS
    # ========================================================
    print(f"\n📊 Cluster {cluster_id} - SHAP Table (Clustered Features):\n")
    display(shap_df_clustered)

    # ========================================================
    # 8️⃣ OPTIONAL: TOP FEATURES
    # ========================================================
    top_features = shap_df.mean(axis=1).sort_values(ascending=False).head(10)

    print(f"\n🔥 Top 10 Features (Cluster {cluster_id}):\n")
    print(top_features)

In [ ]:
#============================================================
#🔶 2️⃣ EMBEDDING GEOMETRY (UMAP + centroid movement)
#============================================================
from umap import UMAP
import pandas as pd

# ---- Flatten embeddings ----
embeddings = fused.detach().cpu().numpy()  # [T, N, D]
T, N, D = embeddings.shape

emb_flat = embeddings.reshape(-1, D)

# ---- Create labels for plotting ----
age_labels = np.repeat(age_list, N)
cluster_flat = cluster_labels.reshape(-1)

# ---- UMAP projection ----
umap = UMAP(n_components=2, random_state=42)
emb_2d = umap.fit_transform(emb_flat)

# ---- Plot UMAP ----
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x=emb_2d[:, 0],
    y=emb_2d[:, 1],
    hue=cluster_flat,
    style=age_labels,
    palette="tab10",
    alpha=0.7
)
plt.title("UMAP of Embeddings (Cluster + Age)")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

'''
The 2D projection of the learned embeddings using UMAP reveals a well-structured latent space in which patients
organize into clearly separated clusters corresponding to distinct disease stages. The minimal overlap between
clusters indicates that the model has learned discriminative representations that capture meaningful differences
in clinical profiles. Within each cluster, the embeddings are not randomly scattered but instead form structured
patterns, suggesting the presence of intra-stage variability and potential sub-phenotypes. Importantly,
the distribution of age-specific markers across the embedding space shows a coherent directional shift,
implying that patient representations evolve smoothly with age. This continuity supports the interpretation
of disease progression as a gradual trajectory through the latent space rather than abrupt transitions between
discrete states. Additionally, the presence of intermediate regions between clusters suggests transitional stages,
where patients exhibit mixed characteristics of adjacent disease states. Overall, this geometric organization
indicates that the model captures both the separability of disease stages and the temporal dynamics underlying
their progression. small detached groups (e.g., bottom cluster)
Isolated embedding regions may represent rare phenotypes or atypical progression patterns.

“The UMAP visualization reveals a structured embedding space with clear separation between major groups.
A distinct cluster is observed on the right-hand side, well isolated from the remaining samples, indicating strong dissimilarity.
In contrast, the left-hand region contains multiple clusters forming a continuum, including a compact subgroup and more elongated,
partially overlapping distributions, suggesting gradual transitions between these samples. Additionally, sub-clustering within
the main groups indicates the presence of finer-grained structure in the data. Overall, the spatial organization of the embeddings
demonstrates that the model effectively captures meaningful relationships between samples
'''

In [ ]:
# ============================================================
# 🔶 1️⃣ Compute centroids for all clusters at all ages
# ============================================================

centroids = []

for t in range(T):
    for c in np.unique(cluster_labels[t]):

        idx = cluster_labels[t] == c

        # ---- Compute centroid ----
        centroid = embeddings[t][idx].mean(axis=0)
        centroid = np.asarray(centroid).reshape(-1)

        centroids.append({
            "age": age_list[t],
            "cluster": int(c),   # ✅ keep TRUE cluster label
            "centroid": centroid
        })

# ============================================================
# 🔶 2️⃣ Convert to arrays
# ============================================================

centroid_points = np.vstack([x["centroid"] for x in centroids])
centroid_ages = np.array([x["age"] for x in centroids])
centroid_clusters = np.array([x["cluster"] for x in centroids])

print("Centroid shape:", centroid_points.shape)

# ============================================================
# 🔶 3️⃣ Project using UMAP (same model as before)
# ============================================================

centroid_2d = umap.transform(centroid_points)

# ============================================================
# 🔶 4️⃣ Plot trajectories for ALL clusters
# ============================================================

plt.figure(figsize=(10, 8))

for c in np.unique(centroid_clusters):

    idx = np.where(centroid_clusters == c)[0]

    # 🔥 Sort by age (VERY IMPORTANT)
    sorted_idx = idx[np.argsort(centroid_ages[idx])]

    # ---- Plot trajectory ----
    plt.plot(
        centroid_2d[sorted_idx, 0],
        centroid_2d[sorted_idx, 1],
        marker='o',
        label=f'Cluster {c}'
    )

    # ---- Optional: arrows to show direction ----
    for i in range(len(sorted_idx) - 1):
        plt.arrow(
            centroid_2d[sorted_idx[i], 0],
            centroid_2d[sorted_idx[i], 1],
            centroid_2d[sorted_idx[i+1], 0] - centroid_2d[sorted_idx[i], 0],
            centroid_2d[sorted_idx[i+1], 1] - centroid_2d[sorted_idx[i], 1],
            head_width=0.15,
            alpha=0.4,
            length_includes_head=True
        )

plt.title("Centroid Movement Across Age (All Clusters)")
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
#============================================================
#🔶 3️⃣ TRANSITION EXPLANATION (Δfeature → Δembedding)
#============================================================

delta_features = []
delta_embeddings = []
delta_clusters = []

for t in range(T - 1):

    # ---- Feature change ----
    df = x_seq[t+1].cpu().numpy() - x_seq[t].cpu().numpy()  # [N, F]

    # ---- Embedding change ----
    de = embeddings[t+1] - embeddings[t]  # [N, D]
    de_norm = np.linalg.norm(de, axis=1)  # [N]

    # ---- Cluster change (SAFE) ----
    c1 = np.asarray(cluster_labels[t]).reshape(-1)
    c2 = np.asarray(cluster_labels[t+1]).reshape(-1)

    # Ensure same length
    min_len = min(len(c1), len(c2))
    c1 = c1[:min_len]
    c2 = c2[:min_len]

    dc = (c2 != c1).astype(int)  # [N]

    # ---- Align features and embeddings if needed ----
    df = df[:min_len]
    de_norm = de_norm[:min_len]

    delta_features.append(df)
    delta_embeddings.append(de_norm)
    delta_clusters.append(dc)

# ============================================================
# 🔶 Concatenate safely
# ============================================================

delta_features = np.concatenate(delta_features, axis=0)        # [T*N, F]
delta_embeddings = np.concatenate(delta_embeddings, axis=0)    # [T*N]
delta_clusters = np.concatenate(delta_clusters, axis=0)        # [T*N]

print("Shapes:")
print("delta_features:", delta_features.shape)
print("delta_embeddings:", delta_embeddings.shape)
print("delta_clusters:", delta_clusters.shape)
correlations = []

for f in range(delta_features.shape[1]):
    corr = np.corrcoef(delta_features[:, f], delta_embeddings)[0, 1]
    correlations.append(corr)
'''
plt.figure(figsize=(10, 6))
sns.barplot(x=feature_columns, y=correlations)
plt.xticks(rotation=45)
plt.title("Feature Change vs Embedding Shift")
plt.show()
'''
corr_matrix = np.array(correlations).reshape(1, -1)

sns.heatmap(
    corr_matrix,
    annot=True,
    xticklabels=feature_columns,
    yticklabels=["corr"],
    cmap="coolwarm",
    center=0
)
plt.title("Feature Change vs Embedding Shift (Correlation)")
plt.show()
for f in range(delta_features.shape[1]):
    plt.figure()
    sns.boxplot(
        x=delta_clusters,
        y=delta_features[:, f]
    )
    plt.title(f"{feature_columns[f]} vs Cluster Change")
    plt.xlabel("Cluster Changed (0/1)")
    plt.ylabel("Δ Feature")
    plt.show()

In [ ]:
;;;

In [ ]:
"""
'Initialization-based clustering stability'
'''
Fixed the dataset
Repeated K-means with different random seeds
Measured agreement using ARI
Stopped when stability degraded
'''
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score
import pandas as pd
import numpy as np

# ------------------------------
# 1️⃣ Flatten and normalize embeddings
# ------------------------------
T, N, D = embeddings.shape
emb_2d = embeddings.reshape(T * N, D)
emb_2d = StandardScaler().fit_transform(emb_2d)

# ------------------------------
# 2️⃣ Set max k and stability threshold
# ------------------------------
k_max = min(int(np.sqrt(len(emb_2d))), max_k_limit)
stability_threshold = 0.9  # stop if ARI drops below this
n_repeats = 10               # repeat clustering to measure stability
best_stable_k = 2

# ------------------------------
# 3️⃣ Loop over k with stability check
# ------------------------------
for k in range(2, k_max + 1):
    labels_list = []
    for r in range(n_repeats):
        km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=seed+r)
        labels_list.append(km.fit_predict(emb_2d))

    # Compute mean pairwise stability (ARI) between repeats
    aris = []
    for i in range(n_repeats):
        for j in range(i+1, n_repeats):
            aris.append(adjusted_rand_score(labels_list[i], labels_list[j]))
    mean_stability = np.mean(aris)

    print(f"k={k}, mean stability={mean_stability:.3f}")

    # Stop if stability drops below threshold
    if mean_stability < stability_threshold:
        print(f"Stability below {stability_threshold}, stopping at k={k-1}")
        break
    best_stable_k = k

print("Chosen number of clusters based on stability:", best_stable_k)

# ------------------------------
# 4️⃣ Final clustering using stable k
# ------------------------------
final_labels = KMeans(
    n_clusters=best_stable_k,
    init='k-means++',
    n_init=10,
    random_state=seed
).fit_predict(emb_2d)

# ------------------------------
# 5️⃣ Optional: compute metrics
# ------------------------------
id_df["cluster"] = final_labels
ari_hdcat, ari_shoulson = compute_ari(final_labels, id_df.copy(), df_meta)

print("ARI with HD category:", ari_hdcat)
print("ARI with Shoulson:", ari_shoulson)

'''
To determine the number of clusters, we employed a stability-based approach in which k-means clustering was repeated multiple times for each
candidate value of k using different random initializations. Clustering stability was quantified using the adjusted Rand index (ARI) between all pairs of repeated solutions.
The number of clusters was increased incrementally, and the optimal k was selected as the largest value for which stability remained high; specifically,
clustering was terminated when the mean ARI dropped by more than 10% relative to the previous k.


You repeated K-Means for the same
𝑘
k with different random seeds (n_repeats in your code).

You measured the agreement between the repeated clusterings using Adjusted Rand Index (ARI), which is exactly how Ben-Hur et al. quantify cluster stability.

You stopped increasing
𝑘
k when the stability dropped below a threshold (stability_threshold). This “breakpoint” logic is directly from the stability-based cluster validation framework.

✅ So, your implementation is a practical application of stability-based cluster validation:

Reference: Ben-Hur, A., Elisseeff, A., & Guyon, I. (2002). A stability based method for discovering structure in clustered data. Pacific Symposium on Biocomputing.
'''

"""

In [ ]:
'Explain Features contributed to Embeddings'
# ============================================================
# 🔍 Explainability: Feature Importance Over Time
# ============================================================

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import copy
from sklearn.preprocessing import MinMaxScaler

# ============================================================
# 1️⃣ SALIENCY MAP (Over ALL time steps)
# ============================================================

all_saliency = []

for t in range(T):

    # ---- Prepare input with gradients ----
    x_seq_grad = x_seq.clone().detach()

    x_t = x_seq_grad[t].clone().detach().requires_grad_(True)
    x_t.retain_grad()  # 🔥 critical fix

    x_seq_grad[t] = x_t

    # ---- Forward ----
    model.zero_grad()
    fused, _ = model(x_seq_grad, edge_index_seq)

    # ---- Target (embedding magnitude) ----
    target = fused[t].norm(dim=1).mean()

    # ---- Backward ----
    target.backward()

    # ---- Extract saliency ----
    saliency_t = x_t.grad.abs().cpu().numpy()  # [N, F]

    # ---- Aggregate across nodes ----
    saliency_mean = saliency_t.mean(axis=0)  # [F]

    all_saliency.append(saliency_mean)

# ---- Convert to matrix ----
# shape: [T, F] → transpose → [F, T]
saliency_matrix = np.array(all_saliency).T


# ============================================================
# 2️⃣ GUIDED BACKPROPAGATION (Over ALL time steps)
# ============================================================

# ---- Custom Guided ReLU ----
class GuidedReLU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)
        return input.clamp(min=0)

    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        grad_input = grad_output.clone()
        grad_input[input < 0] = 0
        grad_input[grad_output < 0] = 0
        return grad_input


# ---- Replace ReLU safely ----
def replace_relu_with_guided(module):
    for name, mod in module.named_children():
        if isinstance(mod, nn.ReLU):
            module._modules[name] = nn.Sequential()
            module._modules[name].forward = lambda x: GuidedReLU.apply(x)
        else:
            replace_relu_with_guided(mod)


# ---- Clone trained model ----
guided_model = copy.deepcopy(model)
replace_relu_with_guided(guided_model)
guided_model.eval()

guided_all = []

for t in range(T):

    x_seq_grad = x_seq.clone().detach()

    x_t = x_seq_grad[t].clone().detach().requires_grad_(True)
    x_t.retain_grad()

    x_seq_grad[t] = x_t

    # ---- Forward ----
    guided_model.zero_grad()
    fused_g, _ = guided_model(x_seq_grad, edge_index_seq)

    # ---- Target ----
    target_g = fused_g[t].norm(dim=1).mean()

    # ---- Backward ----
    target_g.backward()

    guided_saliency_t = x_t.grad.abs().cpu().numpy()

    guided_all.append(guided_saliency_t.mean(axis=0))

# ---- Convert to matrix ----
guided_matrix = np.array(guided_all).T  # [F, T]


# ============================================================
# 3️⃣ OPTIONAL NORMALIZATION (better visualization)
# ============================================================

scaler = MinMaxScaler()

saliency_matrix_norm = scaler.fit_transform(saliency_matrix)
guided_matrix_norm = scaler.fit_transform(guided_matrix)


# ============================================================
# 4️⃣ VISUALIZATION
# ============================================================

# Saliency with row clustering
# -----------------------------
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(16, 10))
sns.clustermap(
    saliency_matrix_norm,
    cmap="magma",
    xticklabels=age_list,
    yticklabels=feature_columns,
    row_cluster=True,       # 🔹 hierarchical clustering on features
    col_cluster=False,      # keep age/time in order
    linewidths=0.5,
    linecolor='gray',
    figsize=(14, 10)
)
plt.title("Saliency Map with Feature Clustering", pad=50)
plt.show()


# Guided Backprop with row clustering
# -----------------------------
sns.clustermap(
    guided_matrix_norm,
    cmap="magma",
    xticklabels=age_list,
    yticklabels=feature_columns,
    row_cluster=True,       # 🔹 hierarchical clustering on features
    col_cluster=False,
    linewidths=0.5,
    linecolor='gray',
    figsize=(14, 10)
)
plt.title("Guided Backpropagation with Feature Clustering", pad=50)
plt.show()


# In tabular form
#---------------------------------------
import pandas as pd
import seaborn as sns

# ---- Saliency clustermap ----
cg = sns.clustermap(
    saliency_matrix_norm,
    cmap="magma",
    xticklabels=age_list,
    yticklabels=feature_columns,
    row_cluster=True,
    col_cluster=False,
    linewidths=0.5,
    linecolor='gray',
    figsize=(14, 10)
)

#plt.title("Saliency Map with Feature Clustering - Normalized", pad=50)
#plt.show()

# ---- Extract the order of rows (features) after clustering ----
row_order = cg.dendrogram_row.reordered_ind
clustered_features = [feature_columns[i] for i in row_order]

# ---- Rearrange saliency matrix according to clustered features ----
saliency_clustered = saliency_matrix_norm[row_order, :]

# ---- Convert to DataFrame for tabular display ----
df_saliency_clustered = pd.DataFrame(
    saliency_clustered,
    index=clustered_features,   # Features after clustering
    columns=age_list             # Timepoints
)

# ---- Display the table ----
print("Saliency Values (Features × Time) - Clustered Rows")
display(df_saliency_clustered)

In [ ]:
'Explain Features Contribute to Clustering definitions'
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

'SHAP Answers: which features caused a patient to be assigned to a stage ❓'
'“i.e. Which features, at which ages, define each discovered disease stage.”'
# ============================================================
# Calculate Labels for specific number of clusters:
#=============================================================
# Final clustering using best_k
k_val=4
final_kmeans = KMeans(n_clusters=k_val, init='k-means++', n_init=10, random_state=seeds_c)
cluster_labels = final_kmeans.fit_predict(emb_2d)  # shape: [T*N]

id_df["cluster"] = cluster_labels
heatmap_df = id_df.pivot(
    index="subjid",
    columns="age",
    values="cluster"
)
# ============================================================
# 1️⃣ Train surrogate classifier (embeddings → clusters)
# ============================================================
'''
Because KMeans is not explainable in feature space, and SHAP needs a predictive function.
The RandomForestClassifier creates exactly that missing function:
f(embedding) → cluster probability
this what wrapper do:
f(raw features) → embedding → cluster probability
'''
'''
“Given an embedding, what cluster does it belong to?”
This model does NOT see your original features (the raw input data).
It only sees the embedding space.
'''
clf = RandomForestClassifier(n_estimators=100, random_state=seeds_c)
clf.fit(emb_2d, cluster_labels)
pred = clf.predict(emb_2d)
acc = accuracy_score(cluster_labels, pred)

print(f"Surrogate training accuracy: {acc:.4f}")
'''
✔ Interpretation:
≥ 0.95 → Excellent (trust SHAP ✅)
0.85 – 0.95 → Acceptable (be cautious ⚠️)
< 0.85 → Problem (SHAP unreliable ❌)
'''

# ============================================================
# 2️⃣ Wrapper
# ============================================================
# Wrapper answers:
# “If I change feature X, how does cluster assignment change?”
'''
The wrapper connects raw features → embeddings → classifier.
It answers:
“If I change the original input features, how does the predicted cluster change?”
'''
def model_wrapper_features(x_numpy, t_idx):
    x_tensor = torch.tensor(x_numpy, dtype=torch.float)

    x_seq_input = x_tensor.unsqueeze(0)
    edge_idx = [edge_index_seq[t_idx]]

    with torch.no_grad():
        fused, _ = model(x_seq_input, edge_idx)

    emb_flat = fused.reshape(x_tensor.shape[0], -1).cpu().numpy()
    return clf.predict_proba(emb_flat)
'''
predict_proba gives SHAP a smooth “scale” to measure feature impact; Returns probabilities for each class (cluster); if they are 4 clusters, predict_proba returns: [[0.1, 0.7, 0.1, 0.1]] : .
predict gives only a label → too discrete → SHAP can’t quantify contributions well.
'''



'''
15-20% Background sample is sourced from https://arxiv.org/abs/1705.07874;
Following SHAP recommendations, we used a subset of 50 patients as the reference background set for computing feature attributions, representing ~15–20% of the cohort, balancing computational efficiency and stable estimates (Lundberg & Lee, 2017).
'''
# ============================================================
# 3️⃣ Compute SHAP once per timestep (shared across clusters)
# ============================================================
def plot_shap_summary(shap_values, X, feature_names, cluster_id=None, max_display=20):
    """
    Plots SHAP summary plot.

    Parameters:
    - shap_values: SHAP output object (from explainer)
    - X: original input data [N, F]
    - feature_names: list of feature names
    - cluster_id: int (for specific cluster) or None (all classes)
    - max_display: number of top features to show
    """
    import shap
    import numpy as np

    # If multi-class → select one cluster
    if cluster_id is not None:
        values = shap_values.values[:, :, cluster_id]
    else:
        values = shap_values.values  # full multi-class

    # Plot
    shap.summary_plot(
        values,
        X,
        feature_names=feature_names,
        max_display=max_display,
        show=True
    )

shap_values_list = []
for t_idx in range(T):

    X_t = np.array(all_data[t_idx].x)
    '''
    Intuition: SHAP measures how much each feature pushes the model away from the “typical patient” prediction.
    You simply took the 50 random patients at this timestep as the reference population.
    The background in SHAP is your reference population
    It tells SHAP:
    “This is what ‘normal’ or ‘typical’ patients look like.”
    Then, SHAP asks for each patient:
    “How does this patient differ from the typical ones? Which features push the prediction up or down?”
    '''
    # ✅ RANDOM background instead of first 50
    n_bg = min(100, len(X_t))
    idx = np.random.choice(len(X_t), size=n_bg, replace=False)
    background = X_t[idx]

    explainer = shap.Explainer(
        lambda x: model_wrapper_features(x, t_idx),
        background
    )

    shap_values = explainer(X_t)  # [N, F, C]
    shap_values_list.append(shap_values)
     # ✅ Plot for each cluster
    for cluster_id in range(k_val):
        print(f"\nSHAP Summary — Time {t_idx}, Cluster {cluster_id}")
        plot_shap_summary(
            shap_values,
            X_t,
            feature_columns,
            cluster_id=cluster_id
        )
    # background-based Shapley calculation (subset sampling), not permutation.

# ============================================================
# 4️⃣ LOOP OVER ALL CLUSTERS
# ============================================================


for cluster_id in range(k_val):

    print(f"\n================ CLUSTER {cluster_id} ================\n")

    shap_time_feature = []

    for t_idx, sv in enumerate(shap_values_list):

        # select cluster
        sv_cluster = sv.values[:, :, cluster_id]  # [N, F]

        # aggregate across nodes
        sv_mean = np.mean(np.abs(sv_cluster), axis=0)  # [F]

        shap_time_feature.append(sv_mean)

    # [T, F] → [F, T]
    shap_matrix = np.array(shap_time_feature).T

    # ========================================================
    # Normalize
    # ========================================================
    scaler = MinMaxScaler()
    shap_matrix_norm = scaler.fit_transform(shap_matrix)

    # ========================================================
    # Create DataFrame (IMPORTANT)
    # ========================================================
    shap_df = pd.DataFrame(
        shap_matrix_norm,
        index=feature_columns,
        columns=age_list
    )

    # ========================================================
    # 5️⃣ Heatmap (row clustered)
    # ========================================================
    sns.set(font_scale=0.9)
    sns.set_style("whitegrid")

    g = sns.clustermap(
        shap_df,
        cmap="RdBu_r",
        row_cluster=True,
        col_cluster=False,
        linewidths=0.5,
        figsize=(14, 8)
    )

    plt.title(f"SHAP Feature Importance Over Time (Cluster {cluster_id})", pad=20)
    plt.xlabel("Age (Time)")
    plt.ylabel("Features")
    plt.show()

    # ========================================================
    # 6️⃣ Extract row-clustered order
    # ========================================================
    row_order = g.dendrogram_row.reordered_ind
    shap_df_clustered = shap_df.iloc[row_order]

    # ========================================================
    # 7️⃣ PRINT TABULAR RESULTS
    # ========================================================
    print(f"\n📊 Cluster {cluster_id} - SHAP Table (Clustered Features):\n")
    display(shap_df_clustered)

    # ========================================================
    # 8️⃣ OPTIONAL: TOP FEATURES
    # ========================================================
    top_features = shap_df.mean(axis=1).sort_values(ascending=False).head(10)

    print(f"\n🔥 Top 10 Features (Cluster {cluster_id}):\n")
    print(top_features)



'''
Explaining SHAP Summary PLOT

🧠 What a SHAP Summary Plot Shows

A SHAP summary plot is a visualization of:
Feature attribution across all samples
Each dot = one patient
Each row = one feature

🔍 How to Read It (Step-by-step)
1️⃣ Y-axis → Feature Importance (Global ranking)
Features are sorted top → bottom
Top = most important for predicting that cluster
  👉 Interpretation:
  “These features define this disease stage the most”
2️⃣ X-axis → SHAP Value (Impact on prediction)
Right(+) → pushes prediction toward the cluster
Left (–) → pushes prediction away from the cluster
  👉 For your case:
  “Does this feature make a patient more likely to be in Cluster K?”

3️⃣ Color → Feature Value
🔴 Red = high feature value
🔵 Blue = low feature value
  👉Examples:
  Case A: Red on the RIGHT
  High value → increases probability of cluster
  Case B: Red on the LEFT
  High value → decreases probability of cluster
'''

In [ ]:
from matplotlib.colors import ListedColormap
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# ============================================================
# 1️⃣ Merge metadata into id_df (so order stays identical)
# ============================================================
ddf=df1.copy(deep=True)
stage_df = id_df.merge(
    ddf[["subjid", "age", "diagconf", "tfcscore"]],
    on=["subjid", "age"],
    how="left"
)

# ============================================================
# 2️⃣ Create Manifest Variable (DCL-based)
# ============================================================

# 1 = Manifest (diagconf == 4)
# 0 = Premanifest
stage_df["manifest"] = np.where(stage_df["diagconf"] == 4, 1, 0)

# ============================================================
# 3️⃣ Compute Shoulson Stage
# ============================================================

stage_df["shoulson_stage"] = stage_df["tfcscore"].apply(shoulson_stage)

# ============================================================
# 4️⃣ Pivot EXACTLY like cluster heatmap
# ============================================================

heatmap_manifest = stage_df.pivot(
    index="subjid",
    columns="age",
    values="manifest"
)

heatmap_shoulson = stage_df.pivot(
    index="subjid",
    columns="age",
    values="shoulson_stage"
)

# ============================================================
# 5️⃣ Get row order from CLUSTER heatmap
# ============================================================

cluster_heatmap = id_df.pivot(
    index="subjid",
    columns="age",
    values="cluster"
)

g = sns.clustermap(
    cluster_heatmap,
    cmap=sns.color_palette("crest", as_cmap=True),
    linewidths=0.5,
    linecolor="gray",
    row_cluster=True,
    col_cluster=False,
    figsize=(12, 8)
)

row_order = cluster_heatmap.index[g.dendrogram_row.reordered_ind]
plt.close()  # close this plot (we only needed ordering)

# Apply SAME order to new heatmaps
heatmap_manifest = heatmap_manifest.loc[row_order]
heatmap_shoulson = heatmap_shoulson.loc[row_order]

# ============================================================
# 6️⃣ Plot Manifest Heatmap (Binary)
# ============================================================

cmap=sns.color_palette("crest", as_cmap=True)
# Blue = Premanifest
# Red = Manifest

sns.clustermap(
    heatmap_manifest,
    cmap=cmap,
    linewidths=0.5,
    linecolor="gray",
    row_cluster=True,   # IMPORTANT: keep same order
    col_cluster=False,
    figsize=(15, 10)
)

plt.suptitle("Manifest vs Premanifest (Based on DCL)", y=1.02)
plt.show()


# ============================================================
# 7️⃣ Plot Shoulson Heatmap
# ============================================================

sns.clustermap(
    heatmap_shoulson.fillna(-1),
    cmap=cmap,
    linewidths=0.5,
    linecolor="gray",
    row_cluster=True,   # Changed to False to prevent re-clustering with NaNs
    col_cluster=False,
    figsize=(15, 10)
)

plt.suptitle("Shoulson Stage Across Age", y=1.02)
plt.show()

In [ ]:
'''
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import davies_bouldin_score, silhouette_score, adjusted_rand_score
import numpy as np

# -----------------------------
# Flatten and normalize embeddings
# -----------------------------
T, N, D = embeddings.shape
emb_2d = embeddings.reshape(T * N, D)
emb_2d = StandardScaler().fit_transform(emb_2d)

# -----------------------------
# Parameters
# -----------------------------
k_range = range(2, 11)       # candidate k
n_bootstrap = 20             # number of bootstrap resamples
random_seed = 42
np.random.seed(random_seed)

results = []

# -----------------------------
# Bootstrap stability evaluation
# -----------------------------
for k in k_range:
    dbi_list = []
    ari_list = []
    labels_boot = []

    for b in range(n_bootstrap):
        # Sample with replacement
        idx = np.random.choice(len(emb_2d), size=len(emb_2d), replace=True)
        sample = emb_2d[idx]

        # Cluster
        km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=random_seed+b)
        labels = km.fit_predict(sample)
        labels_boot.append(labels)

        # Davies-Bouldin index
        if len(np.unique(labels)) > 1:
            dbi = davies_bouldin_score(sample, labels)
            dbi_list.append(dbi)

    # Compute pairwise ARI stability across bootstraps
    aris = []
    for i in range(len(labels_boot)):
        for j in range(i+1, len(labels_boot)):
            min_len = min(len(labels_boot[i]), len(labels_boot[j]))
            aris.append(adjusted_rand_score(labels_boot[i][:min_len], labels_boot[j][:min_len]))

    mean_dbi = np.mean(dbi_list)
    std_dbi = np.std(dbi_list)
    mean_ari = np.mean(aris)

    results.append({
        "k": k,
        "mean_dbi": mean_dbi,
        "std_dbi": std_dbi,
        "mean_ari": mean_ari
    })

# -----------------------------
# Convert to DataFrame
# -----------------------------
import pandas as pd
results_df = pd.DataFrame(results)

# -----------------------------
# Choose optimal k
# -----------------------------
# 1. Most stable: highest mean ARI (stability)
# 2. Optionally, also low DBI (good clustering quality)
best_k = results_df.sort_values(["mean_ari", "mean_dbi"], ascending=[False, True]).iloc[0]["k"]

print("Bootstrap stability results:")
print(results_df)
print("\nChosen k (consensus/bootstrap stability):", best_k)
'''

In [ ]:
'Selecting the best dimention: https://www.nature.com/articles/s41467-021-23795-5?utm_source=chatgpt.com'
"""
import itertools
import numpy as np

# Candidate dimensions for i, j, k
I_candidates = [ 4, 8, 16, 32, 64]
J_candidates = [ 4, 8, 16, 32, 64]
K_candidates = [ 4, 8, 16, 32, 64]

epsilon = 0.05
seed = seeds_c + 1

# Dictionary to store losses
loss_table = {}

# ----------------------------
# Grid search over i, j, k
# ----------------------------
for i, j, k in itertools.product(I_candidates, J_candidates, K_candidates):
    print(f"Training model with i={i}, j={j}, k={k}")
    set_seed(seed)

    # Train model and get both fused + reconstructed
    set_seed(seed)
    model = SpatioTemporalEncoder(F_dim_features, i, j, k)
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    model.train()
    for _ in range(500):
        opt.zero_grad()
        fused, reconstructed = model(x_seq, edge_index_seq)
        loss_train = F.mse_loss(reconstructed[mask], x_seq[mask], reduction='sum') / mask.sum()
        loss_train.backward()
        opt.step()

    # Eval and get reconstructed
    model.eval()
    with torch.no_grad():
        _, reconstructed = model(x_seq, edge_index_seq)

    # Compute loss
    x_seq_np = x_seq.cpu().numpy()
    reconstructed_np = reconstructed.cpu().numpy()
    loss = np.mean((reconstructed_np[mask.cpu().numpy()] - x_seq_np[mask.cpu().numpy()])**2)
    loss_table[(i,j,k)] = loss



# ----------------------------
# Find global plateau
# ----------------------------
L_inf = min(loss_table.values())

'''
1/Try all possible embedding sizes (i,j,k).
2/Compute reconstruction loss for each.
3/Identify the best achievable loss L_inf.
4/Keep all dimensions that are within epsilon of L_inf.

Choose the smallest dimensions among them → minimal size for sufficient quality.
We don’t always need the absolute smallest loss;
slightly higher losses are fine if it means smaller embeddings.
We define epsilon (like 0.05).
'''

# ----------------------------
# Find minimal (i,j,k) within epsilon of plateau
# ----------------------------
best_combo = None
for combo in sorted(loss_table.keys(), key=lambda x: (x[0]+x[1]+x[2], x[2], x[1], x[0])):
    # sort by sum of dimensions (minimal total size first)
    if loss_table[combo] - L_inf < epsilon:
        best_combo = combo
        break

print("\n📊 Loss-based selection of i, j, k")
for combo, loss in loss_table.items():
    print(f"i={combo[0]}, j={combo[1]}, k={combo[2]} -> Loss={loss:.6f}")

print(f"\n✅ Best dimensions within epsilon={epsilon}: i={best_combo[0]}, j={best_combo[1]}, k={best_combo[2]}")
print(f"Corresponding loss = {loss_table[best_combo]:.6f}, plateau = {L_inf:.6f}")

"""


In [ ]:
"""
good_dims = [
    combo for combo, loss in loss_table.items()
    if loss <= L_inf + epsilon
]
print(good_dims)

print(f"\nPlateau loss L_inf = {L_inf:.6f}")
print("Dimensions within epsilon:")
for combo in good_dims:
    print(combo, "loss =", loss_table[combo])


'✅ STEP 3: Multi-seed consistency test (ONLY on good dims)'
num_seeds = 5
variance_table = {}

for i, j, k in good_dims:
    print(f"\nConsistency test: i={i}, j={j}, k={k}")
    embeddings = []

    for seed in range(num_seeds):
        set_seed(seed)

        model = SpatioTemporalEncoder(F_dim_features, i, j, k)
        opt = torch.optim.Adam(model.parameters(), lr=0.01)

        model.train()
        for _ in range(500):
            opt.zero_grad()
            fused, reconstructed = model(x_seq, edge_index_seq)
            loss = F.mse_loss(
                reconstructed[mask],
                x_seq[mask],
                reduction="sum"
            ) / mask.sum()
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            fused, _ = model(x_seq, edge_index_seq)

        embeddings.append(fused.cpu().numpy())

    embeddings = np.stack(embeddings)  # (seeds, T, N, D)
    variance = np.mean(np.var(embeddings, axis=0))
    variance_table[(i, j, k)] = variance


    '''
>>> Keep all dimensions that are within epsilon of L_inf.
>>> Choose the smallest dimensions among them → minimal size for sufficient quality.
    '''

#The smallest one is @ 16, 4, 16.
"""

In [ ]:
"""
best_combo = min(variance_table, key=variance_table.get)

print("\n📊 Final results")
for combo in variance_table:
    print(
        f"i={combo[0]}, j={combo[1]}, k={combo[2]} | "
        f"loss={loss_table[combo]:.6f} | "
        f"variance={variance_table[combo]:.6f}"
    )

print("\n✅ BEST DIMENSION (most consistent on plateau)")
print(
    f"i={best_combo[0]}, j={best_combo[1]}, k={best_combo[2]}"
)
print(
    f"loss={loss_table[best_combo]:.6f}, "
    f"variance={variance_table[best_combo]:.6f}"
)
"""

In [ ]:
"""
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')

i_vals, j_vals, k_vals, losses = zip(*[(i,j,k,loss) for (i,j,k), loss in loss_table.items()])

p = ax.scatter(i_vals, j_vals, k_vals, c=losses, cmap='viridis', s=100)
fig.colorbar(p, label='Loss')
ax.set_xlabel('i dimension')
ax.set_ylabel('j dimension')
ax.set_zlabel('k dimension')
ax.set_title('Loss values for all i,j,k combinations')
plt.show()
"""

In [ ]:
'clusters features visualization'

In [ ]:
# Flatten features to match clustering shape

T, N, F = x_seq.shape  # x_seq is [T, N, F]
flat_features = x_seq.reshape(T*N, F)  # [T*N, F]

# Create DataFrame with subjid, age, cluster, and features
feature_df = pd.DataFrame(flat_features, columns=feature_columns)
feature_df['subjid'] = np.repeat(all_nodes_global, T)
feature_df['age'] = np.tile(selected_ages, N)
feature_df['cluster'] = cluster_labels

#feature_df=id_df.copy(deep=True)
# Swap cluster labels: 0 -> 1 and 1 -> 0 # ****************************************SWIPE**********************************
id_df['cluster'] = id_df['cluster'].replace({0: 1, 1: 0})


In [ ]:
id_df['cluster'] = id_df['cluster'].replace({0: 1, 1: 0})

# ============================================================
# Premanifest vs Manifest Cluster Heatmaps with Converters
#   ✔ Legend shows actual cluster numbers
#   ✔ Always pre/always man + converters included
#   ✔ Row clustering
#   ✔ White for missing
#   ✔ Fancy colors
# ============================================================

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# ------------------------------------------------------------
# 1️⃣ Merge DCL
# ------------------------------------------------------------
stage_df = id_df.merge(
    df1[["subjid", "age", "diagconf"]],
    on=["subjid", "age"],
    how="left"
)

# ------------------------------------------------------------
# 2️⃣ Pivot clusters and DCL
# ------------------------------------------------------------
cluster_pivot = stage_df.pivot(index="subjid", columns="age", values="cluster")
dcl_pivot     = stage_df.pivot(index="subjid", columns="age", values="diagconf")

cluster_pivot = cluster_pivot.replace(-10, np.nan)

# ------------------------------------------------------------
# 3️⃣ Identify subject types
# ------------------------------------------------------------
subject_status = stage_df.groupby("subjid")["diagconf"].agg(
    lambda x: (np.any(x < 4), np.any(x == 4))
)

always_pre  = subject_status[subject_status.apply(lambda x: x[0] and not x[1])].index
always_man  = subject_status[subject_status.apply(lambda x: x[1] and not x[0])].index
converters  = subject_status[subject_status.apply(lambda x: x[0] and x[1])].index

# ------------------------------------------------------------
# 4️⃣ Subset matrices
# ------------------------------------------------------------
# Premanifest heatmap
heatmap_pre = cluster_pivot.loc[np.concatenate([always_pre, converters])]
heatmap_pre = heatmap_pre.where(dcl_pivot.loc[heatmap_pre.index] < 4)

# Manifest heatmap
heatmap_man = cluster_pivot.loc[np.concatenate([always_man, converters])]
heatmap_man = heatmap_man.where(dcl_pivot.loc[heatmap_man.index] == 4)

# ------------------------------------------------------------
# 5️⃣ Masks and fill for clustering
# ------------------------------------------------------------
mask_pre = heatmap_pre.isna()
mask_man = heatmap_man.isna()

heatmap_pre_filled = heatmap_pre.fillna(-1)
heatmap_man_filled = heatmap_man.fillna(-1)

# ------------------------------------------------------------
# 6️⃣ Colormap
# ------------------------------------------------------------
cluster_numbers = np.unique(id_df["cluster"])
n_clusters = len(cluster_numbers)
fancy_colors = sns.color_palette("husl", n_clusters)
cluster_cmap = ListedColormap(fancy_colors)

# Legend shows actual cluster numbers
legend_handles = [
    Patch(facecolor=fancy_colors[i], label=f"Stage {cluster_numbers[i]}")
    for i in range(n_clusters)
]

# ------------------------------------------------------------
# 7️⃣ Premanifest Heatmap
# ------------------------------------------------------------
g1 = sns.clustermap(
    heatmap_pre_filled,
    cmap=cluster_cmap,
    mask=mask_pre,
    vmin=0,
    vmax=n_clusters - 1,
    linewidths=0.4,
    linecolor="lightgray",
    row_cluster=True,
    col_cluster=False,
    figsize=(14, 10),
    cbar=True
)

# Remove extra left label
#g1.ax_heatmap.set_ylabel('')
#g1.ax_heatmap.set_yticks([])  # optional: remove all y-axis ticks

# Legend
g1.ax_heatmap.legend(
    handles=legend_handles,
    title="", fontsize=24,
    bbox_to_anchor=(0.5, -0.1),  # adjust below the heatmap
    loc="upper center",
    ncol=n_clusters
)
g1.ax_heatmap.set_xlabel('Age', fontsize=24) # Changed set_xlabels to set_xlabel
g1.ax_heatmap.set_xticklabels(
    g1.ax_heatmap.get_xticklabels(),
    fontsize=24
)
g1.ax_heatmap.set_ylabel('Patients IDs', fontsize=24)

plt.suptitle("Discovered Stages — Premanifest (DCL < 4 )", y=1.02,fontsize=24)
plt.show()


# ------------------------------------------------------------
# 8️⃣ Manifest Heatmap
# ------------------------------------------------------------
g2 = sns.clustermap(
    heatmap_man_filled,
    cmap=cluster_cmap,
    mask=mask_man,
    vmin=0,
    vmax=n_clusters - 1,
    linewidths=0.4,
    linecolor="lightgray",
    row_cluster=True,
    col_cluster=False,
    figsize=(14, 10),
    cbar=True
)

# Remove extra left label
#g2.ax_heatmap.set_ylabel('')
#g2.ax_heatmap.set_yticks([])  # optional

# Legend
g2.ax_heatmap.legend(
    handles=legend_handles,
    title="", fontsize=24,
    bbox_to_anchor=(0.5, -0.1),
    loc="upper center",
    ncol=n_clusters
)
g2.ax_heatmap.set_xlabel('Age', fontsize=24) # Changed set_xlabels to set_xlabel
g2.ax_heatmap.set_xticklabels(
    g2.ax_heatmap.get_xticklabels(),
    fontsize=16
)
g2.ax_heatmap.set_ylabel('Patients IDs', fontsize=24)


plt.suptitle("Discovered Stages — Manifest (DCL = 4)", y=1.02,fontsize=24)
plt.show()
# Increase heatmap column names font size



In [ ]:
cluster_summary = feature_df.groupby('cluster')[feature_columns].describe().T
print(cluster_summary)


In [ ]:
feature_df['cluster'] = feature_df['cluster'].replace({0: 1, 1: 0})


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

cluster_means = feature_df.groupby('cluster')[feature_columns].mean()

# Define feature groups exactly as you provided
motor_features = [
    "ocularh","ocularv","sacinith","sacinitv","sacvelh","sacvelv",
    "dysarth","tongue","fingtapr","fingtapl","prosupr","prosupl",
    "luria","rigarmr","rigarml","brady",
    "dysttrnk","dystrue","dystlue","dystrle","dystlle",
    "chorface","chorbol","chortrnk","chorrue","chorlue","chorrle","chorlle",
    "gait","tandem","retropls"
]

functional_features = [
    "indepscl","carelevl","adl","chores","finances","occupatn"
]

cognitive_features = [
    "mmsetotal","sit1","verflt05","swrt1","scnt1","verfct5","sdmt1"
]

# Combine features in desired order: cognitive → functional → motor
ordered_features = cognitive_features + functional_features + motor_features

# Filter and reorder cluster means
cluster_means_ordered = cluster_means[ordered_features]

# Plot heatmap with reordered features
plt.figure(figsize=(14,8))
sns.heatmap(cluster_means_ordered.T, cmap='coolwarm', annot=False, fmt=".2f")
plt.xlabel("Discovered Clusters", fontsize=16)
plt.ylabel("Features", fontsize=16)
plt.title("Mean Feature Values per Cluster (Grouped by Cognitive, Functional, Motor)", fontsize=16)
plt.show()



In [ ]:
'Assess features normality: Option A: Shapiro–Wilk (Most Common)'
'''
If the p-value of the Shapiro–Wilk test is greater than 0.05, it generally means:
        Fail to reject the null hypothesis.

Interpretation
Null hypothesis (H₀): The data follows a normal distribution.
Alternative hypothesis (H₁): The data is not normally distributed.
'''
from scipy.stats import shapiro

# Initialize dictionary to store results
normality_results = {}

for feature in ordered_features:
    data = feature_df[feature].dropna()

    # Shapiro-Wilk test
    stat, p_value = shapiro(data)

    normality_results[feature] = {
        "W-statistic": stat,
        "p-value": p_value,
        "Normal?": "Yes" if p_value > 0.05 else "No"
    }

# Convert to DataFrame
normality_df = pd.DataFrame(normality_results).T
print(normality_df)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from scipy.stats import spearmanr

'VIS1: SPIDER PLOT PER CLINICAL DOMAIN'
domains = {
    "Oculomotor": ["ocularh","ocularv","sacinith","sacinitv","sacvelh","sacvelv"],
    "Motor": [
        "dysarth","tongue","fingtapr","fingtapl","prosupr","prosupl",
        "luria","rigarmr","rigarml","brady","gait","tandem","retropls"
    ],
    "Chorea": [
        "chorface","chorbol","chortrnk","chorrue","chorlue","chorrle","chorlle"
    ],
    "Dystonia": [
        "dysttrnk","dystrue","dystlue","dystrle","dystlle"
    ],
    "Functional": ["indepscl","carelevl","adl","chores","finances","occupatn"],
    "Cognitive": [
        "mmsetotal","sit1","verflt05","swrt1",
        "scnt1","verfct5","sdmt1"
    ]
}


'''
domain_df = pd.DataFrame({
    domain: feature_df_z[feats].mean(axis=1)
    for domain, feats in domains.items()
})

domain_df["cluster"] = feature_df["cluster"]
'''
# Define posthoc_features earlier for merging
posthoc_features = ["capscore","tfcscore","motscore","diagconf","hdcat","cag", 'hdiss_stage_imp']

# Merge posthoc_features into feature_df
# Assuming df_meta still contains the original posthoc_features with subjid and age
feature_df_all = feature_df.merge(df_meta[['subjid', 'age'] + posthoc_features], on=['subjid', 'age'], how='left')


################################################
# Z-score features first
scaler = StandardScaler()
feature_df_z = feature_df_all.copy()
feature_df_z[feature_columns] = scaler.fit_transform(feature_df[feature_columns])



In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import kruskal
from scipy.stats import f_oneway

# Features to test
features_to_test = ["diagconf", "motscore", "tfcscore"]

# Container for results
stats_results = []

for var in features_to_test:

    # Drop missing values
    df_temp = feature_df_all[["cluster", var]].dropna()

    # Split into groups (4 clusters)
    groups = [df_temp[df_temp["cluster"] == c][var]
              for c in sorted(df_temp["cluster"].unique())]

    # --- Kruskal-Wallis test (non-parametric) ---
    stat, p_value = kruskal(*groups)

    stats_results.append({
        "Variable": var,
        "Test": "Kruskal-Wallis",
        "Statistic": stat,
        "p-value": p_value
    })

# Convert to DataFrame
stats_results_df = pd.DataFrame(stats_results)

# Add significance flag
stats_results_df["Significant (p<0.05)"] = stats_results_df["p-value"] < 0.05

print("\n=== Statistical Test Results Across Clusters ===")
print(stats_results_df)


##########################################################
# OPTIONAL: If you also want ANOVA (for comparison)
##########################################################

anova_results = []

for var in features_to_test:

    df_temp = feature_df_all[["cluster", var]].dropna()
    groups = [df_temp[df_temp["cluster"] == c][var]
              for c in sorted(df_temp["cluster"].unique())]

    stat, p_value = f_oneway(*groups)

    anova_results.append({
        "Variable": var,
        "Test": "ANOVA",
        "Statistic": stat,
        "p-value": p_value
    })

anova_results_df = pd.DataFrame(anova_results)

print("\n=== ANOVA Results (optional) ===")
print(anova_results_df)


# -----------------------------
# 2️⃣ Mean + 95% CI function
# -----------------------------
def mean_ci(series):
    series = series.dropna()
    n = len(series)

    if n == 0:
        return pd.Series({
            "mean": np.nan,
            "ci_lower": np.nan,
            "ci_upper": np.nan,
            "n": 0
        })

    mean = series.mean()
    se = series.std(ddof=1) / np.sqrt(n)
    ci = 1.96 * se

    return pd.Series({
        "mean": mean,
        "ci_lower": mean - ci,
        "ci_upper": mean + ci,
        "n": n
    })


# -----------------------------
# 3️⃣ Compute summary per cluster
# -----------------------------
summary = (
    feature_df_all
    .groupby("cluster")[vars_of_interest]
    .apply(lambda df: df.apply(mean_ci))
)

summary = summary.unstack(level=1)


# -----------------------------
# 4️⃣ Format as "mean [CI]"
# -----------------------------
def format_mean_ci(row):
    if pd.isna(row["mean"]):
        return "NA"
    return f"{row['mean']:.2f} [{row['ci_lower']:.2f}–{row['ci_upper']:.2f}]"

formatted = summary.copy()

for var in vars_of_interest:
    formatted[(var, "formatted")] = summary[var].apply(format_mean_ci, axis=1)

final_table = formatted.xs("formatted", level=1, axis=1)


# -----------------------------
# 5️⃣ Output final table
# -----------------------------
print("\nCluster-wise Mean [95% CI]:\n")
print(final_table)


# -----------------------------
# (Optional) Save results
# -----------------------------
# final_table.to_csv("cluster_summary_table.csv")
# kruskal_df.to_csv("kruskal_results.csv")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# =========================================
# RADAR – MEAN FEATURE VALUES PER STAGE WITH 95% CI
# =========================================


# 2) Compute mean and 95% CI for each feature within each stage
stage_stats = {}

for stage in sorted(feature_df_z["cluster"].dropna().unique()):
    subset = feature_df_z[feature_df_z["cluster"] == stage]

    mean_vals = subset[feature_columns].mean()
    sem_vals = subset[feature_columns].std(ddof=1) / np.sqrt(len(subset))
    ci95 = 1.96 * sem_vals

    stage_stats[stage] = pd.DataFrame({
        'mean': mean_vals,
        'ci95': ci95
    })

# 3) Radar geometry
labels = feature_columns
num_vars = len(labels)
angles = np.linspace(0, 2*np.pi, num_vars, endpoint=False)
angles = np.append(angles, angles[0])  # close circle

# 4) Figure
fig, ax = plt.subplots(figsize=(16, 16), subplot_kw=dict(polar=True))

# 5) Radial limits
all_means = pd.concat([stage_stats[s]['mean'] for s in stage_stats], axis=0)
r_min = all_means.min() - 0.5
r_max = all_means.max() + 0.5
ax.set_ylim(r_min, r_max)

colors = plt.cm.plasma_r(np.linspace(0, 1, len(stage_stats)))

# 6) Plot each stage with 95% CI
for (stage, stats), color in zip(stage_stats.items(), colors):
    mean_vals = stats['mean'].values
    ci95_vals = stats['ci95'].values

    mean_closed = np.append(mean_vals, mean_vals[0])
    upper_vals = np.append(mean_vals + ci95_vals,
                           (mean_vals + ci95_vals)[0])
    lower_vals = np.append(mean_vals - ci95_vals,
                           (mean_vals - ci95_vals)[0])

    ax.plot(angles, mean_closed,
            linewidth=2,
            color=color,
            label=f"Stage {stage}")

    ax.fill_between(angles, lower_vals, upper_vals,
                    alpha=0.15,
                    color=color)

# =========================================
# Vertical labels along radial line WITH SQUARE BRACKETS
# =========================================
ax.set_xticks(angles[:-1])        # draw radial lines
ax.set_xticklabels([])           # hide default text

label_radius = r_max + 0.9      # distance from center to label

for angle, label in zip(angles[:-1], labels):
    angle_deg = np.degrees(angle)

    # Flip text if on left half of the circle
    if 90 < angle_deg < 270:
        rotation = angle_deg + 180
    else:
        rotation = angle_deg

    ax.text(
        angle,
        label_radius,
        label,
        size=26,
        rotation=rotation,
        rotation_mode='anchor',
        ha='center',
        va='center',
        clip_on=False
    )

# =========================================
# Title & legend
ax.set_title(
    "Mean Z-Scored Features per URL-STFN-based Stage with 95% CI\n\n\n\n",
    fontsize=24,
    weight='bold',
    pad=40
)

ax.legend(loc="upper right",
          bbox_to_anchor=(1.25, 1.1),
          fontsize=24)

plt.tight_layout()
plt.show()

In [ ]:
'''

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (14, 6)

#---------------------------------------------------
# 0️⃣ Prepare stages
#---------------------------------------------------
def shoulson_stage(tfc):
    if tfc >= 11: return "Stage I"
    elif tfc >= 7: return "Stage II"
    elif tfc >= 3: return "Stage III"
    elif tfc >= 1: return "Stage IV"
    else: return "Stage V"

feature_df_all["shoulson_stage"] = feature_df_all["tfcscore"].apply(shoulson_stage)
feature_df_all["manifest_status"] = np.where(feature_df_all["diagconf"]==4, "Manifest", "Premanifest")
feature_df_all["discovered_stage"] = feature_df_all["cluster"].astype(str)

#---------------------------------------------------
# 1️⃣ Function to compute percentages per age
#---------------------------------------------------
def percent_per_age(df, col):
    pct = (
        df.groupby("age")[col]
        .value_counts(normalize=True)
        .mul(100)
        .rename("percent")
        .reset_index()
    )
    return pct

dcl_pct = percent_per_age(feature_df_all, "manifest_status")
shoulson_pct = percent_per_age(feature_df_all, "shoulson_stage")
cluster_pct = percent_per_age(feature_df_all, "discovered_stage")

#---------------------------------------------------
# 2️⃣ Function to plot side-by-side 100% stacked bars with labels
#---------------------------------------------------
def plot_side_by_side_with_labels(existing_pct, discovered_pct, existing_col, title, existing_cmap="coolwarm", discovered_cmap="tab10"):
    ages = sorted(feature_df["age"].unique())
    width = 0.4  # width of each bar

    fig, ax = plt.subplots(figsize=(14,6))

    # Pivot data
    existing_pivot = existing_pct.pivot(index="age", columns=existing_col, values="percent").fillna(0)
    discovered_pivot = discovered_pct.pivot(index="age", columns="discovered_stage", values="percent").fillna(0)

    # Plot bars side by side
    for i, age in enumerate(ages):
        # Existing stage bar
        bottom_existing = 0
        for j, col in enumerate(existing_pivot.columns):
            val = existing_pivot.loc[age, col]
            ax.bar(i - width/2, val, width, bottom=bottom_existing,
                   color=plt.cm.get_cmap(existing_cmap)(j / len(existing_pivot.columns)), edgecolor='k')
            if val > 3:  # only label if segment is big enough
                ax.text(i - width/2, bottom_existing + val/2, f"{val:.1f}%", ha='center', va='center', fontsize=8, color='white', fontweight='bold')
            bottom_existing += val

        # Discovered stage bar
        bottom_discovered = 0
        for j, col in enumerate(discovered_pivot.columns):
            val = discovered_pivot.loc[age, col]
            ax.bar(i + width/2, val, width, bottom=bottom_discovered,
                   color=plt.cm.get_cmap(discovered_cmap)(j / len(discovered_pivot.columns)), edgecolor='k')
            if val > 3:
                ax.text(i + width/2, bottom_discovered + val/2, f"{val:.1f}%", ha='center', va='center', fontsize=8, color='white', fontweight='bold')
            bottom_discovered += val

    # X-axis
    ax.set_xticks(range(len(ages)))
    ax.set_xticklabels(ages)
    ax.set_ylabel("Percentage (%)")
    ax.set_xlabel("Age")
    ax.set_ylim(0,100)
    ax.set_title(title)

    # Legends with clear titles
    existing_handles = [plt.Rectangle((0,0),1,1,color=plt.cm.get_cmap(existing_cmap)(i/len(existing_pivot.columns))) for i in range(len(existing_pivot.columns))]
    discovered_handles = [plt.Rectangle((0,0),1,1,color=plt.cm.get_cmap(discovered_cmap)(i/len(discovered_pivot.columns))) for i in range(len(discovered_pivot.columns))]

    ax.legend(existing_handles + discovered_handles,
              [f"{col}" for col in existing_pivot.columns] + [f"{col}" for col in discovered_pivot.columns],
              title="Existing Stage vs Discovered Stage", bbox_to_anchor=(1.05,1))
    plt.show()

#---------------------------------------------------
# 3️⃣ Plot DCL vs Discovered Cluster
#---------------------------------------------------
plot_side_by_side_with_labels(dcl_pct, cluster_pct, "manifest_status",
                              "Manifest / Premanifest vs Discovered Cluster per Age",
                              existing_cmap="coolwarm", discovered_cmap="tab10")

#---------------------------------------------------
# 4️⃣ Plot Shoulson vs Discovered Cluster
#---------------------------------------------------
plot_side_by_side_with_labels(shoulson_pct, cluster_pct, "shoulson_stage",
                              "Shoulson Stage vs Discovered Cluster per Age",
                              existing_cmap="viridis", discovered_cmap="tab10")
'''

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# =========================================
# RADAR – MEAN FEATURE VALUES PER STAGE WITH 95% CI
# =========================================

#---------------------------------------------------
#  Prepare stages
#---------------------------------------------------
def shoulson_stage(tfc):
    if tfc >= 11:
        return 'Stage I'
    elif 7 <= tfc <= 10:
        return 'Stage II'
    elif 4 <= tfc <= 6:
        return 'Stage III'
    elif 1 <= tfc <= 3:
        return 'Stage IV'
    elif tfc == 0:
        return 'Stage V'
    else:
        return np.nan

feature_df_z["shoulson_stage"] = feature_df_all["tfcscore"].apply(shoulson_stage)
feature_df_z["manifest_status"] = np.where(feature_df_all["diagconf"]==4, "Manifest", "Premanifest")
#feature_df_z["discovered_stage"] = feature_df_all["cluster"].astype(str)

# 2) Compute mean and 95% CI for each feature within each stage
stage_stats = {}

for stage in sorted(feature_df_z["manifest_status"].dropna().unique()):
    subset = feature_df_z[feature_df_z["manifest_status"] == stage]

    mean_vals = subset[feature_columns].mean()
    sem_vals = subset[feature_columns].std(ddof=1) / np.sqrt(len(subset))
    ci95 = 1.96 * sem_vals

    stage_stats[stage] = pd.DataFrame({
        'mean': mean_vals,
        'ci95': ci95
    })

# 3) Radar geometry
labels = feature_columns
num_vars = len(labels)
angles = np.linspace(0, 2*np.pi, num_vars, endpoint=False)
angles = np.append(angles, angles[0])  # close circle

# 4) Figure
fig, ax = plt.subplots(figsize=(16, 16), subplot_kw=dict(polar=True))

# 5) Radial limits
all_means = pd.concat([stage_stats[s]['mean'] for s in stage_stats], axis=0)
r_min = all_means.min() - 0.5
r_max = all_means.max() + 0.5
ax.set_ylim(r_min, r_max)

colors = plt.cm.plasma_r(np.linspace(0, 1, len(stage_stats)))

# 6) Plot each stage with 95% CI
for (stage, stats), color in zip(stage_stats.items(), colors):
    mean_vals = stats['mean'].values
    ci95_vals = stats['ci95'].values

    mean_closed = np.append(mean_vals, mean_vals[0])
    upper_vals = np.append(mean_vals + ci95_vals,
                           (mean_vals + ci95_vals)[0])
    lower_vals = np.append(mean_vals - ci95_vals,
                           (mean_vals - ci95_vals)[0])

    ax.plot(angles, mean_closed,
            linewidth=2,
            color=color,
            label=f"Stage {stage}")

    ax.fill_between(angles, lower_vals, upper_vals,
                    alpha=0.15,
                    color=color)

# =========================================
# Vertical labels along radial line WITH SQUARE BRACKETS
# =========================================
ax.set_xticks(angles[:-1])        # draw radial lines
ax.set_xticklabels([])           # hide default text

label_radius = r_max + 0.2        # distance from center to label

for angle, label in zip(angles[:-1], labels):
    angle_deg = np.degrees(angle)

    # Flip text if on left half of the circle
    if 90 < angle_deg < 270:
        rotation = angle_deg + 180
    else:
        rotation = angle_deg

    ax.text(
        angle,
        label_radius,
        label,
        size=26,
        rotation=rotation,
        rotation_mode='anchor',
        ha='center',
        va='center',
        clip_on=False
    )

# =========================================
# Title & legend
ax.set_title(
    "Mean Z-Scored Features per DCL-based Stage with 95% CI\n\n\n\n",
    fontsize=20,
    weight='bold',
    pad=40
)

ax.legend(loc="upper right",
          bbox_to_anchor=(1.25, 1.1),
          fontsize=20)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# =========================================
# RADAR – MEAN FEATURE VALUES PER STAGE WITH 95% CI
# =========================================

# 2) Compute mean and 95% CI for each feature within each stage
stage_stats = {}

for stage in sorted(feature_df_z["shoulson_stage"].dropna().unique()):
    subset = feature_df_z[feature_df_z["shoulson_stage"] == stage]

    mean_vals = subset[feature_columns].mean()
    sem_vals = subset[feature_columns].std(ddof=1) / np.sqrt(len(subset))
    ci95 = 1.96 * sem_vals

    stage_stats[stage] = pd.DataFrame({
        'mean': mean_vals,
        'ci95': ci95
    })

# 3) Radar geometry
labels = feature_columns
num_vars = len(labels)
angles = np.linspace(0, 2*np.pi, num_vars, endpoint=False)
angles = np.append(angles, angles[0])  # close circle

# 4) Figure
fig, ax = plt.subplots(figsize=(16, 16), subplot_kw=dict(polar=True))

# 5) Radial limits
all_means = pd.concat([stage_stats[s]['mean'] for s in stage_stats], axis=0)
r_min = all_means.min() - 0.5
r_max = all_means.max() + 0.5
ax.set_ylim(r_min, r_max)

colors = plt.cm.plasma_r(np.linspace(0, 1, len(stage_stats)))

# 6) Plot each stage with 95% CI
for (stage, stats), color in zip(stage_stats.items(), colors):
    mean_vals = stats['mean'].values
    ci95_vals = stats['ci95'].values

    mean_closed = np.append(mean_vals, mean_vals[0])
    upper_vals = np.append(mean_vals + ci95_vals,
                           (mean_vals + ci95_vals)[0])
    lower_vals = np.append(mean_vals - ci95_vals,
                           (mean_vals - ci95_vals)[0])

    ax.plot(angles, mean_closed,
            linewidth=2,
            color=color,
            label=f"Stage {stage}")

    ax.fill_between(angles, lower_vals, upper_vals,
                    alpha=0.15,
                    color=color)

# =========================================
# Vertical labels along radial line WITH SQUARE BRACKETS
# =========================================
ax.set_xticks(angles[:-1])        # draw radial lines
ax.set_xticklabels([])           # hide default text

label_radius = r_max + 0.3        # distance from center to label

for angle, label in zip(angles[:-1], labels):
    angle_deg = np.degrees(angle)

    # Flip text if on left half of the circle
    if 90 < angle_deg < 270:
        rotation = angle_deg + 180
    else:
        rotation = angle_deg

    ax.text(
        angle,
        label_radius,
        label,
        size=26,
        rotation=rotation,
        rotation_mode='anchor',
        ha='center',
        va='center',
        clip_on=False
    )

# =========================================
# Title & legend
ax.set_title(
    "Mean Z-Scored Features per Shoulson & Fahn stage-based Stage with 95% CI\n\n\n\n",
    fontsize=20,
    weight='bold',
    pad=40
)

ax.legend(loc="upper right",
          bbox_to_anchor=(1.25, 1.1),
          fontsize=20)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# =========================================
# RADAR – MEAN FEATURE VALUES PER STAGE WITH 95% CI
# =========================================


# 2) Compute mean and 95% CI for each feature within each stage
stage_stats = {}

for stage in sorted(feature_df_z["shoulson_stage"].dropna().unique()):
    subset = feature_df_z[feature_df_z["shoulson_stage"] == stage]

    mean_vals = subset[feature_columns].mean()
    sem_vals = subset[feature_columns].std(ddof=1) / np.sqrt(len(subset))
    ci95 = 1.96 * sem_vals

    stage_stats[stage] = pd.DataFrame({
        'mean': mean_vals,
        'ci95': ci95
    })

# 3) Radar geometry
labels = feature_columns
num_vars = len(labels)
angles = np.linspace(0, 2*np.pi, num_vars, endpoint=False)
angles = np.append(angles, angles[0])  # close circle

# 4) Figure
fig, ax = plt.subplots(figsize=(16, 16), subplot_kw=dict(polar=True))

# 5) Radial limits
all_means = pd.concat([stage_stats[s]['mean'] for s in stage_stats], axis=0)
r_min = all_means.min() - 0.5
r_max = all_means.max() + 0.5
ax.set_ylim(r_min, r_max)

colors = plt.cm.plasma_r(np.linspace(0, 1, len(stage_stats)))

# 6) Plot each stage with 95% CI
for (stage, stats), color in zip(stage_stats.items(), colors):
    mean_vals = stats['mean'].values
    ci95_vals = stats['ci95'].values

    mean_closed = np.append(mean_vals, mean_vals[0])
    upper_vals = np.append(mean_vals + ci95_vals,
                           (mean_vals + ci95_vals)[0])
    lower_vals = np.append(mean_vals - ci95_vals,
                           (mean_vals - ci95_vals)[0])

    ax.plot(angles, mean_closed,
            linewidth=2,
            color=color,
            label=f"Stage {stage}")

    ax.fill_between(angles, lower_vals, upper_vals,
                    alpha=0.15,
                    color=color)

# =========================================
# Vertical labels along radial line WITH SQUARE BRACKETS
# =========================================
ax.set_xticks(angles[:-1])        # draw radial lines
ax.set_xticklabels([])           # hide default text

label_radius = r_max + 0.3        # distance from center to label

for angle, label in zip(angles[:-1], labels):
    angle_deg = np.degrees(angle)

    # Flip text if on left half of the circle
    if 90 < angle_deg < 270:
        rotation = angle_deg + 180
    else:
        rotation = angle_deg

    ax.text(
        angle,
        label_radius,
        label,
        size=26,
        rotation=rotation,
        rotation_mode='anchor',
        ha='center',
        va='center',
        clip_on=False
    )

# =========================================
# Title & legend
ax.set_title(
    "Mean Z-Scored Features per Shoulson & Fahn stage-based Stage with 95% CI\n\n\n\n",
    fontsize=20,
    weight='bold',
    pad=40
)

ax.legend(loc="upper right",
          bbox_to_anchor=(1.25, 1.1),
          fontsize=20)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# =========================================
# RADAR – MEAN FEATURE VALUES PER STAGE WITH 95% CI
# =========================================

# 2) Compute mean and 95% CI for each feature within each stage
stage_stats = {}

for stage in sorted(feature_df_z["shoulson_stage"].dropna().unique()):
    subset = feature_df_z[feature_df_z["shoulson_stage"] == stage]

    mean_vals = subset[feature_columns].mean()
    sem_vals = subset[feature_columns].std(ddof=1) / np.sqrt(len(subset))
    ci95 = 1.96 * sem_vals

    stage_stats[stage] = pd.DataFrame({
        'mean': mean_vals,
        'ci95': ci95
    })

# 3) Radar geometry
labels = feature_columns
num_vars = len(labels)
angles = np.linspace(0, 2*np.pi, num_vars, endpoint=False)
angles = np.append(angles, angles[0])  # close circle

# 4) Figure
fig, ax = plt.subplots(figsize=(16, 16), subplot_kw=dict(polar=True))

# 5) Radial limits
all_means = pd.concat([stage_stats[s]['mean'] for s in stage_stats], axis=0)
r_min = all_means.min() - 0.5
r_max = all_means.max() + 0.5
ax.set_ylim(r_min, r_max)

colors = plt.cm.plasma_r(np.linspace(0, 1, len(stage_stats)))

# 6) Plot each stage with 95% CI
for (stage, stats), color in zip(stage_stats.items(), colors):
    mean_vals = stats['mean'].values
    ci95_vals = stats['ci95'].values

    mean_closed = np.append(mean_vals, mean_vals[0])
    upper_vals = np.append(mean_vals + ci95_vals,
                           (mean_vals + ci95_vals)[0])
    lower_vals = np.append(mean_vals - ci95_vals,
                           (mean_vals - ci95_vals)[0])

    ax.plot(angles, mean_closed,
            linewidth=2,
            color=color,
            label=f"Stage {stage}")

    ax.fill_between(angles, lower_vals, upper_vals,
                    alpha=0.15,
                    color=color)

# =========================================
# Vertical labels along radial line WITH SQUARE BRACKETS
# =========================================
ax.set_xticks(angles[:-1])        # draw radial lines
ax.set_xticklabels([])           # hide default text

label_radius = r_max + 0.3        # distance from center to label

for angle, label in zip(angles[:-1], labels):
    angle_deg = np.degrees(angle)

    # Flip text if on left half of the circle
    if 90 < angle_deg < 270:
        rotation = angle_deg + 180
    else:
        rotation = angle_deg

    ax.text(
        angle,
        label_radius,
        label,
        size=26,
        rotation=rotation,
        rotation_mode='anchor',
        ha='center',
        va='center',
        clip_on=False
    )

# =========================================
# Title & legend
ax.set_title(
    "Mean Z-Scored Features per Shoulson & Fahn stage-based Stage with 95% CI\n\n\n\n",
    fontsize=20,
    weight='bold',
    pad=40
)

ax.legend(loc="upper right",
          bbox_to_anchor=(1.25, 1.1),
          fontsize=20)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from scipy.stats import spearmanr

sns.set(style="whitegrid", context="talk")  # beautify seaborn plots
plt.rcParams["figure.figsize"] = (8, 6)

# 2) Compute mean and 95% CI for each feature within each stage
stage_stats = {}

for stage in sorted(feature_df_z["hdiss_stage_imp"].dropna().unique()):
    subset = feature_df_z[feature_df_z["hdiss_stage_imp"] == stage]

    mean_vals = subset[feature_columns].mean()
    sem_vals = subset[feature_columns].std(ddof=1) / np.sqrt(len(subset))
    ci95 = 1.96 * sem_vals

    stage_stats[stage] = pd.DataFrame({
        'mean': mean_vals,
        'ci95': ci95
    })

# 3) Radar geometry
labels = feature_columns
num_vars = len(labels)
angles = np.linspace(0, 2*np.pi, num_vars, endpoint=False)
angles = np.append(angles, angles[0])  # close circle

# 4) Figure
fig, ax = plt.subplots(figsize=(16, 16), subplot_kw=dict(polar=True))

# 5) Radial limits
all_means = pd.concat([stage_stats[s]['mean'] for s in stage_stats], axis=0)
r_min = all_means.min() - 0.5
r_max = all_means.max() + 0.5
ax.set_ylim(r_min, r_max)

colors = plt.cm.plasma_r(np.linspace(0, 1, len(stage_stats)))

# 6) Plot each stage with 95% CI
for (stage, stats), color in zip(stage_stats.items(), colors):
    mean_vals = stats['mean'].values
    ci95_vals = stats['ci95'].values

    mean_closed = np.append(mean_vals, mean_vals[0])
    upper_vals = np.append(mean_vals + ci95_vals,
                           (mean_vals + ci95_vals)[0])
    lower_vals = np.append(mean_vals - ci95_vals,
                           (mean_vals - ci95_vals)[0])

    ax.plot(angles, mean_closed,
            linewidth=2,
            color=color,
            label=f"Stage {stage}")

    ax.fill_between(angles, lower_vals, upper_vals,
                    alpha=0.15,
                    color=color)

# =========================================
# Vertical labels along radial line WITH SQUARE BRACKETS
# =========================================
ax.set_xticks(angles[:-1])        # draw radial lines
ax.set_xticklabels([])           # hide default text

label_radius = r_max + 0.9        # distance from center to label

for angle, label in zip(angles[:-1], labels):
    angle_deg = np.degrees(angle)

    # Flip text if on left half of the circle
    if 90 < angle_deg < 270:
        rotation = angle_deg + 180
    else:
        rotation = angle_deg

    ax.text(
        angle,
        label_radius,
        label,
        size=26,
        rotation=rotation,
        rotation_mode='anchor',
        ha='center',
        va='center',
        clip_on=False
    )

# =========================================
# Title & legend
ax.set_title(
    "Mean Z-Scored Features per HDIIS stage-based Stage with 95% CI\n\n\n\n",
    fontsize=20,
    weight='bold',
    pad=40
)

ax.legend(loc="upper right",
          bbox_to_anchor=(1.25, 1.1),
          fontsize=20)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# =========================================
# RADAR – MEAN FEATURE VALUES PER STAGE WITH 95% CI
# =========================================


# 2) Compute mean and 95% CI for each feature within each stage
stage_stats = {}

for stage in sorted(feature_df_z["cluster"].dropna().unique()):
    subset = feature_df_z[feature_df_z["cluster"] == stage]

    mean_vals = subset[feature_columns].mean()
    sem_vals = subset[feature_columns].std(ddof=1) / np.sqrt(len(subset))
    ci95 = 1.96 * sem_vals

    stage_stats[stage] = pd.DataFrame({
        'mean': mean_vals,
        'ci95': ci95
    })

# 3) Radar geometry
labels = feature_columns
num_vars = len(labels)
angles = np.linspace(0, 2*np.pi, num_vars, endpoint=False)
angles = np.append(angles, angles[0])  # close circle

# 4) Figure
fig, ax = plt.subplots(figsize=(16, 16), subplot_kw=dict(polar=True))

# 5) Radial limits
all_means = pd.concat([stage_stats[s]['mean'] for s in stage_stats], axis=0)
r_min = all_means.min() - 0.5
r_max = all_means.max() + 0.5
ax.set_ylim(r_min, r_max)

colors = plt.cm.plasma_r(np.linspace(0, 1, len(stage_stats)))

# 6) Plot each stage with 95% CI
for (stage, stats), color in zip(stage_stats.items(), colors):
    mean_vals = stats['mean'].values
    ci95_vals = stats['ci95'].values

    mean_closed = np.append(mean_vals, mean_vals[0])
    upper_vals = np.append(mean_vals + ci95_vals,
                           (mean_vals + ci95_vals)[0])
    lower_vals = np.append(mean_vals - ci95_vals,
                           (mean_vals - ci95_vals)[0])

    ax.plot(angles, mean_closed,
            linewidth=2,
            color=color,
            label=f"Stage {stage}")

    ax.fill_between(angles, lower_vals, upper_vals,
                    alpha=0.15,
                    color=color)

# =========================================
# Vertical labels along radial line WITH SQUARE BRACKETS
# =========================================
ax.set_xticks(angles[:-1])        # draw radial lines
ax.set_xticklabels([])           # hide default text

label_radius = r_max + 0.6        # distance from center to label

for angle, label in zip(angles[:-1], labels):
    angle_deg = np.degrees(angle)

    # Flip text if on left half of the circle
    if 90 < angle_deg < 270:
        rotation = angle_deg + 180
    else:
        rotation = angle_deg

    ax.text(
        angle,
        label_radius,
        label,
        size=26,
        rotation=rotation,
        rotation_mode='anchor',
        ha='center',
        va='center',
        clip_on=False
    )

# =========================================
# Title & legend
ax.set_title(
    "Mean Z-Scored Features per URL-STFN-based Stage with 95% CI\n\n\n\n",
    fontsize=24,
    weight='bold',
    pad=40
)

ax.legend(loc="upper right",
          bbox_to_anchor=(1.25, 1.1),
          fontsize=24)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# ================================
# RADAR – MEAN FEATURE VALUES PER STAGE
# ================================


# 2) Compute mean of each feature within each stage
stage_means = {}
for stage in sorted(feature_df_z["manifest_status"].dropna().unique()):
    subset = feature_df_z[feature_df_z["manifest_status"] == stage]
    stage_means[stage] = subset[feature_columns].mean()
stage_means_df = pd.DataFrame(stage_means).T  # rows = stage, cols = features

# 3) Radar geometry
labels = feature_columns
num_vars = len(labels)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False)
angles = np.append(angles, angles[0])  # close circle

fig, ax = plt.subplots(figsize=(12, 12), subplot_kw=dict(polar=True))

# radial limits
r_min = stage_means_df.min().min() - 0.5
r_max = stage_means_df.max().max() + 0.5
ax.set_ylim(r_min, r_max)

# color map
colors = plt.cm.plasma(np.linspace(0, 1, len(stage_means_df)))

# 4) Plot each stage
for (stage, row), color in zip(stage_means_df.iterrows(), colors):
    values = np.append(row.values, row.values[0])
    ax.plot(angles, values, linewidth=2, color=color, label=f"Stage {stage}")

# 5) Rotate feature labels to align with their axes
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=14)

for label, angle in zip(ax.get_xticklabels(), angles[:-1]):
    # Rotate the label so its baseline aligns with the radial axis
    angle_deg = np.degrees(angle)

    # Labels along top-right quadrant (0-90) or bottom-right (270-360) stay as is
    # Labels along left half (90-270) are rotated 180 to align with axis
    if 90 < angle_deg < 270:
        rotation = angle_deg + 180
    else:
        rotation = angle_deg

    label.set_rotation(rotation)
    label.set_horizontalalignment('center')
    label.set_verticalalignment('center')
    label.set_rotation_mode('anchor')

ax.set_title("Radar Plot – Mean Z-Scored Features per DCL-Based Stages\n",
             fontsize=16, pad=20)

ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1))
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from scipy.stats import sem

# =========================================
# RADAR – MEAN FEATURE VALUES PER STAGE + 95% CI
# =========================================


# 2) Compute mean and 95% CI per stage
stage_stats = {}

for stage in sorted(feature_df_z["manifest_status"].dropna().unique()):
    subset = feature_df_z[feature_df_z["manifest_status"] == stage]

    mean_vals = subset[feature_columns].mean()
    sem_vals = subset[feature_columns].apply(sem)
    ci95 = 1.96 * sem_vals

    stage_stats[stage] = {
        "mean": mean_vals,
        "lower": mean_vals - ci95,
        "upper": mean_vals + ci95
    }

# 3) Radar geometry
labels = feature_columns
num_vars = len(labels)

angles = np.linspace(0, 2*np.pi, num_vars, endpoint=False)
angles = np.concatenate((angles, [angles[0]]))

fig, ax = plt.subplots(figsize=(12, 12), subplot_kw=dict(polar=True))

# Keep radial scaling consistent
all_means = np.concatenate([stage_stats[s]["mean"].values for s in stage_stats])
r_min = all_means.min() - 0.5
r_max = all_means.max() + 0.5
ax.set_ylim(r_min, r_max)

ax.grid(alpha=0.3)

# 4) Plot each stage
colors = plt.cm.plasma(np.linspace(0, 1, len(stage_stats)))

for (stage, stats), color in zip(stage_stats.items(), colors):

    mean_vals = np.append(stats["mean"].values, stats["mean"].values[0])
    lower_vals = np.append(stats["lower"].values, stats["lower"].values[0])
    upper_vals = np.append(stats["upper"].values, stats["upper"].values[0])

    ax.plot(angles, mean_vals,
            linewidth=2,
            color=color,
            label=f"Stage {stage}")

    ax.fill_between(angles,
                    lower_vals,
                    upper_vals,
                    color=color,
                    alpha=0.20)

# 5) Angle-based label rotation (clean circular style)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=11)

for label, angle in zip(ax.get_xticklabels(), angles[:-1]):
    angle_deg = np.degrees(angle)

    # Rotate label according to angle
    label.set_rotation(angle_deg)

    # Flip labels on left half for readability
    if 90 < angle_deg < 270:
        label.set_rotation(angle_deg + 180)
        label.set_horizontalalignment('right')
    else:
        label.set_horizontalalignment('left')

    label.set_verticalalignment('center')
    label.set_rotation_mode('anchor')

# Optional: push labels outward slightly
ax.tick_params(pad=15)

# 6) Title & legend
ax.set_title(
    "Radar Plot – Mean Z-Scored Features per DCL-Based Stages (95% CI)\n",
    fontsize=14,
    pad=20
)

ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1))

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# =========================================
# RADAR – MEAN FEATURE VALUES PER STAGE
# =========================================


stage_means = {}

for stage in sorted(feature_df_z["cluster"].dropna().unique()):
    subset = feature_df_z[feature_df_z["cluster"] == stage]
    stage_means[stage] = subset[feature_columns].mean()

stage_means_df = pd.DataFrame(stage_means).T  # rows = stage, cols = features

# 3) Radar geometry
labels = feature_columns
num_vars = len(labels)

angles = np.linspace(0, 2*np.pi, num_vars, endpoint=False)
angles = np.append(angles, angles[0])

fig, ax = plt.subplots(figsize=(12, 12), subplot_kw=dict(polar=True))

# radial limits
r_min = stage_means_df.min().min() - 0.5
r_max = stage_means_df.max().max() + 0.5
ax.set_ylim(r_min, r_max)

# color map
colors = plt.cm.plasma(np.linspace(0, 1, len(stage_means_df)))

# 4) Plot each stage mean profile
for (stage, row), color in zip(stage_means_df.iterrows(), colors):
    values = row.values
    values = np.append(values, values[0])

    ax.plot(angles, values,
            linewidth=2,
            color=color,
            label=f"Stage {stage}")

    ax.fill(angles, values,
            alpha=0.15,
            color=color)

# 5) Formatting
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=14)

ax.set_title("Radar Plot – Mean Z-Scored Features per Discover Stage\n",
             fontsize=14, pad=20)

ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1))

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Example: domains
domains = {
    "Oculomotor": ["ocularh","ocularv","sacinith","sacinitv","sacvelh","sacvelv"],
    "Motor": [
        "dysarth","tongue","fingtapr","fingtapl","prosupr","prosupl",
        "luria","rigarmr","rigarml","brady","gait","tandem","retropls"
    ],
    "Chorea": [
        "chorface","chorbol","chortrnk","chorrue","chorlue","chorrle","chorlle"
    ],
    "Dystonia": [
        "dysttrnk","dystrue","dystlue","dystrle","dystlle"
    ],
    "Functional": ["indepscl","carelevl","adl","chores","finances","occupatn"],
    "Cognitive": [
        "mmsetotal","sit1","verflt05","swrt1",
        "scnt1","verfct5","sdmt1"
    ]
}

# Make sure age column exists
df1['age'] = df1['age'].astype(int)

# Create a DataFrame to store missing % by age
missing_by_age = pd.DataFrame()

for domain, features in domains.items():
    for feat in features:
        if feat in df1.columns:
            temp = df1.groupby('age')[feat].apply(lambda x: x.isna().mean() * 100).reset_index()
            temp.columns = ['age', 'MissingPercent']
            temp['Feature'] = feat
            temp['Domain'] = domain
            missing_by_age = pd.concat([missing_by_age, temp], ignore_index=True)

# Pivot for heatmap: ages as rows, features as columns
heatmap_df = missing_by_age.pivot(index='age', columns='Feature', values='MissingPercent')

# Plot
plt.figure(figsize=(20,8))  # widen the figure to fit all features
sns.heatmap(heatmap_df, cmap='Reds', cbar_kws={'label': 'Missing (%)'})

plt.title('Feature Missingness by Age')
plt.ylabel('Age')
plt.xlabel('Feature')

# Show all x-axis labels, rotated for readability
plt.xticks(ticks=range(len(heatmap_df.columns)),
           labels=heatmap_df.columns,
           rotation=90,  # rotate labels vertically
           ha='center')  # horizontal alignment

plt.tight_layout()  # adjust layout so labels fit
plt.show()


In [ ]:
'ASSESS features significant difference across discovered clusters'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
from scipy.stats import kruskal

# ================================
# 1️⃣ Define Feature Groups
# ================================

motor_features = [
    "ocularh","ocularv","sacinith","sacinitv","sacvelh","sacvelv",
    "dysarth","tongue","fingtapr","fingtapl","prosupr","prosupl",
    "luria","rigarmr","rigarml","brady",
    "dysttrnk","dystrue","dystlue","dystrle","dystlle",
    "chorface","chorbol","chortrnk","chorrue","chorlue","chorrle","chorlle",
    "gait","tandem","retropls"
]
functional_features = [
    "indepscl","carelevl","adl","chores","finances","occupatn"
]
cognitive_features = [
    "mmsetotal","sit1","verflt05","swrt1","scnt1","verfct5","sdmt1"
]
posthoc_features = ["capscore","tfcscore","motscore","diagconf","hdcat",'hdiss_stage_imp']

# Features used for clustering
cluster_features = [f for f in motor_features + functional_features + cognitive_features]

print(f"Testing {len(cluster_features)} features across {feature_df['cluster'].nunique()} clusters")


# ================================
# 2️⃣ Kruskal-Wallis Testing
# ================================

results = []

for feat in cluster_features:
    groups = [group[feat].values for name, group in feature_df.groupby("cluster")]
    stat, p = kruskal(*groups)
    results.append((feat, stat, p))

results_df = pd.DataFrame(results, columns=["feature", "statistic", "p_value"])

# Bonferroni correction
results_df["p_adj"] = results_df["p_value"] * len(results_df)
results_df["significant"] = results_df["p_adj"] < 0.05

results_df = results_df.sort_values("p_adj")

print("\nSignificant features after Bonferroni correction:")
print(results_df[results_df["significant"]][["feature","p_adj"]])


# ================================
# 3️⃣ Get All Significant Features
# ================================

significant_feats = results_df[results_df["significant"]]["feature"].tolist()

print(f"\nTotal significant features: {len(significant_feats)}")


# ================================
# 4️⃣ Plotting Function
# ================================

def plot_feature_group(feature_list, title):

    feats = [f for f in feature_list if f in significant_feats]

    if len(feats) == 0:
        print(f"No significant features in {title}")
        return

    n_feats = len(feats)
    n_cols = 4
    n_rows = math.ceil(n_feats / n_cols)

    fig, axs = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    axs = axs.flatten()

    for i, feat in enumerate(feats):
        sns.boxplot(
            x="cluster",
            y=feat,
            data=feature_df,
            palette="husl",
            ax=axs[i]
        )

        sns.stripplot(
            x="cluster",
            y=feat,
            data=feature_df,
            color="k",
            size=3,
            alpha=0.4,
            ax=axs[i]
        )

        axs[i].set_title(feat)
        axs[i].set_xlabel("Cluster")

    # Remove empty axes
    for j in range(i+1, len(axs)):
        fig.delaxes(axs[j])

    fig.suptitle(title, fontsize=18)
    plt.tight_layout()
    plt.show()


# ================================
# 5️⃣ Generate Figures
# ================================

plot_feature_group(motor_features, "Motor Features (Significant Only)")
plot_feature_group(cognitive_features, "Cognitive Features (Significant Only)")
plot_feature_group(functional_features, "Functional Features (Significant Only)")


In [ ]:
'For actual features values'
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

# =================================
# 1️⃣ Define Feature Groups
# =================================
motor_features = [
    "ocularh","ocularv","sacinith","sacinitv","sacvelh","sacvelv",
    "dysarth","tongue","fingtapr","fingtapl","prosupr","prosupl",
    "luria","rigarmr","rigarml","brady",
    "dysttrnk","dystrue","dystlue","dystrle","dystlle",
    "chorface","chorbol","chortrnk","chorrue","chorlue","chorrle","chorlle",
    "gait","tandem","retropls"
]

functional_features = [
    "indepscl","carelevl","adl","chores","finances","occupatn"
]

cognitive_features = [
    "mmsetotal","sit1","verflt05","swrt1","scnt1","verfct5","sdmt1"
]

cluster_features = motor_features + functional_features + cognitive_features+posthoc_features

# =================================
# 2️⃣ Merge raw features with ID info
# =================================
# id_df should contain: subjid, age, cluster
merged_df = pd.merge(
    df1[['subjid', 'age'] + cluster_features],  # raw features
    id_df[['subjid', 'age', 'cluster']],        # cluster labels
    on=['subjid', 'age'],
    how='inner'
)
merged_df['cluster'] = merged_df['cluster'].replace({0: 1, 1: 0})

print(f"Merged dataset shape: {merged_df.shape}")
print(f"Clusters present: {merged_df['cluster'].unique()}")

# =================================
# 3️⃣ Plotting Function (RAW VALUES)
# =================================
def plot_feature_group_raw(df, feature_list, title):
    feats = [f for f in feature_list if f in df.columns]
    n_feats = len(feats)
    n_cols = 4
    n_rows = math.ceil(n_feats / n_cols)

    fig, axs = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    axs = axs.flatten()

    for i, feat in enumerate(feats):
        df_plot = df[['cluster', feat]].dropna()
        df_plot[feat] = df_plot[feat].astype(float)

        # Violin plot
        sns.violinplot(
            x="cluster",
            y=feat,
            data=df_plot,
            palette="husl",
            inner="quartile",
            cut=0,
            ax=axs[i]
        )

        # Overlay raw points
        sns.stripplot(
            x="cluster",
            y=feat,
            data=df_plot,
            color="k",
            size=2,
            alpha=0.4,
            ax=axs[i]
        )

        axs[i].set_title(feat, fontsize=18)
        axs[i].set_xlabel("Cluster")
        axs[i].set_ylabel("Raw Value")

    # Remove empty axes
    for j in range(i+1, len(axs)):
        fig.delaxes(axs[j])

    fig.suptitle(title, fontsize=18)
    plt.tight_layout()
    plt.show()

# =================================
# 4️⃣ Generate Figures
# =================================
plot_feature_group_raw(merged_df, motor_features, "Motor Features (Raw Values) Across Clusters")
plot_feature_group_raw(merged_df, cognitive_features, "Cognitive Features (Raw Values) Across Clusters")
plot_feature_group_raw(merged_df, functional_features, "Functional Features (Raw Values) Across Clusters")

In [ ]:
'Normality Check for actual features per discovered cluster'
import pandas as pd
from scipy.stats import shapiro

# -----------------------------
# 1️⃣ Specify features and cluster column
# -----------------------------
# cluster_features: all numeric clinical features used for clustering
# merged_df: your dataset after merging df1 with id_df on subjid & age
# Assume cluster labels are in column 'cluster'

# -----------------------------
# 2️⃣ Normality testing per feature per cluster
# -----------------------------
normality_results = []

for feat in cluster_features:
    for cl in merged_df['cluster'].unique():
        values = merged_df.loc[merged_df['cluster'] == cl, feat].dropna()

        # Skip very small groups
        if len(values) < 3:
            continue

        stat, p = shapiro(values)  # Shapiro-Wilk test
        normality_results.append({
            'feature': feat,
            'cluster': cl,
            'W_stat': stat,
            'p_value': p,
            'is_normal': p > 0.05  # True if normally distributed
        })

# -----------------------------
# 3️⃣ Convert to DataFrame
# -----------------------------
normality_df = pd.DataFrame(normality_results)

# Optional: pivot table for easy reading
normality_pivot = normality_df.pivot(index='feature', columns='cluster', values='is_normal')
normality_pivot = normality_pivot.fillna(False)  # mark missing as False

# -----------------------------
# 4️⃣ Display results
# -----------------------------
print("Normality per feature per cluster (True = normal, False = not normal):")
print(normality_pivot)

# Summary: how many clusters are normal per feature
summary = normality_df.groupby('feature')['is_normal'].sum().reset_index()
summary.rename(columns={'is_normal': 'num_clusters_normal'}, inplace=True)

print("\nSummary: Number of clusters where each feature is normal:")
print(summary)#
normality_pivot

In [ ]:
'MEADIAN + IQR (Q1-Q3) for actual features'

import pandas as pd
import numpy as np

# Display settings (so all columns appear in console)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Ensure numeric types
features = kept_features
merged_df[features] = merged_df[features].apply(pd.to_numeric, errors='coerce')
merged_df['tfcscore'] = pd.to_numeric(merged_df['tfcscore'], errors='coerce')
merged_df['diagconf'] = pd.to_numeric(merged_df['diagconf'], errors='coerce')
merged_df['hdiss_stage_imp'] = pd.to_numeric(merged_df['hdiss_stage_imp'], errors='coerce')

# ---------------------------
# Calculate Shoulson Stage
# ---------------------------
def assign_shoulson_stage(tfc):
    if tfc >= 11:
        return 'Stage I'
    elif 7 <= tfc <= 10:
        return 'Stage II'
    elif 4 <= tfc <= 6:
        return 'Stage III'
    elif 1 <= tfc <= 3:
        return 'Stage IV'
    elif tfc == 0:
        return 'Stage V'
    else:
        return np.nan

merged_df['shoulson_stage'] = merged_df['tfcscore'].apply(assign_shoulson_stage)

# ---------------------------
# DCL-based stage
# ---------------------------
merged_df['manifest_status'] = np.where(
    merged_df['diagconf'] == 4,
    'Manifest',
    'Premanifest'
)

# ---------------------------
# Helper function: Median ± IQR
# ---------------------------
def median_iqr_str(vals):
    if len(vals) == 0:
        return np.nan
    q1 = vals.quantile(0.25)
    median = vals.median()
    q3 = vals.quantile(0.75)
    return f"{median:.2f} ± [{q1:.2f}–{q3:.2f}]"

# ---------------------------
# Define grouping systems
# ---------------------------
groupings = {
    'manifest_status': ['Manifest', 'Premanifest'],
    'cluster': sorted(merged_df['cluster'].dropna().unique()),
    'hdiss_stage_imp': sorted(merged_df['hdiss_stage_imp'].dropna().unique()),
    'shoulson_stage': ['Stage I','Stage II','Stage III','Stage IV','Stage V']
}

# ---------------------------
# Compute tables for each staging system
# ---------------------------
dfs = {}

for group_name, categories in groupings.items():

    rows = []

    for feat in features:

        row = {'feature': feat}

        for cat in categories:

            vals = merged_df.loc[
                merged_df[group_name] == cat, feat
            ].dropna()

            row[cat] = median_iqr_str(vals)

        rows.append(row)

    dfs[group_name] = pd.DataFrame(rows).set_index('feature')

# ---------------------------
# Rename columns clearly
# ---------------------------
dfs['manifest_status'].columns = [f"DCL {c}" for c in dfs['manifest_status'].columns]
dfs['cluster'].columns = [f"Cluster {c}" for c in dfs['cluster'].columns]
dfs['hdiss_stage_imp'].columns = [f"HDISS {c}" for c in dfs['hdiss_stage_imp'].columns]
dfs['shoulson_stage'].columns = [f"Shoulson {c}" for c in dfs['shoulson_stage'].columns]

# ---------------------------
# Combine all staging systems
# ---------------------------
combined_df = pd.concat(dfs.values(), axis=1)

# ---------------------------
# Desired column order
# ---------------------------
desired_col_order = [

    "DCL Manifest",
    "DCL Premanifest",

    *[f"Cluster {c}" for c in groupings['cluster']],

    *[f"HDISS {c}" for c in groupings['hdiss_stage_imp']],

    *[f"Shoulson {s}" for s in groupings['shoulson_stage']]
]

desired_col_order = [
    col for col in desired_col_order
    if col in combined_df.columns
]

combined_df = combined_df[desired_col_order]

# ---------------------------
# Display result
# ---------------------------
print(combined_df)

# Optional export
# combined_df.to_csv("median_iqr_features_all_stages_wide.csv")

combined_df

In [ ]:
'MEADIAN + IQR (Q1-Q3) for standarized features'

import pandas as pd
import numpy as np

# Display settings (so all columns appear in console)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)


# ---------------------------
# Standardize features (Z-score)
# ---------------------------
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

merged_df_z = merged_df.copy()
merged_df_z[features] = scaler.fit_transform(merged_df[features])


# Ensure numeric types
features = kept_features
merged_df_z[features] = merged_df_z[features].apply(pd.to_numeric, errors='coerce')
merged_df_z['tfcscore'] = pd.to_numeric(merged_df_z['tfcscore'], errors='coerce')
merged_df_z['diagconf'] = pd.to_numeric(merged_df_z['diagconf'], errors='coerce')
merged_df_z['hdiss_stage_imp'] = pd.to_numeric(merged_df_z['hdiss_stage_imp'], errors='coerce')

# ---------------------------
# Calculate Shoulson Stage
# ---------------------------
def assign_shoulson_stage(tfc):
    if tfc >= 11:
        return 'Stage I'
    elif 7 <= tfc <= 10:
        return 'Stage II'
    elif 4 <= tfc <= 6:
        return 'Stage III'
    elif 1 <= tfc <= 3:
        return 'Stage IV'
    elif tfc == 0:
        return 'Stage V'
    else:
        return np.nan

merged_df_z['shoulson_stage'] = merged_df_z['tfcscore'].apply(assign_shoulson_stage)

# ---------------------------
# DCL-based stage
# ---------------------------
merged_df_z['manifest_status'] = np.where(
    merged_df_z['diagconf'] == 4,
    'Manifest',
    'Premanifest'
)

# ---------------------------
# Helper function: Median ± IQR
# ---------------------------
def median_iqr_str(vals):
    if len(vals) == 0:
        return np.nan
    q1 = vals.quantile(0.25)
    median = vals.median()
    q3 = vals.quantile(0.75)
    return f"{median:.2f} ± [{q1:.2f}–{q3:.2f}]"

# ---------------------------
# Define grouping systems
# ---------------------------
groupings = {
    'manifest_status': ['Manifest', 'Premanifest'],
    'cluster': sorted(merged_df_z['cluster'].dropna().unique()),
    'hdiss_stage_imp': sorted(merged_df_z['hdiss_stage_imp'].dropna().unique()),
    'shoulson_stage': ['Stage I','Stage II','Stage III','Stage IV','Stage V']
}

# ---------------------------
# Compute tables for each staging system
# ---------------------------
dfs = {}

for group_name, categories in groupings.items():

    rows = []

    for feat in features:

        row = {'feature': feat}

        for cat in categories:

            vals = merged_df_z.loc[
                merged_df_z[group_name] == cat, feat
            ].dropna()

            row[cat] = median_iqr_str(vals)

        rows.append(row)

    dfs[group_name] = pd.DataFrame(rows).set_index('feature')

# ---------------------------
# Rename columns clearly
# ---------------------------
dfs['manifest_status'].columns = [f"DCL {c}" for c in dfs['manifest_status'].columns]
dfs['cluster'].columns = [f"Cluster {c}" for c in dfs['cluster'].columns]
dfs['hdiss_stage_imp'].columns = [f"HDISS {c}" for c in dfs['hdiss_stage_imp'].columns]
dfs['shoulson_stage'].columns = [f"Shoulson {c}" for c in dfs['shoulson_stage'].columns]

# ---------------------------
# Combine all staging systems
# ---------------------------
combined_df = pd.concat(dfs.values(), axis=1)

# ---------------------------
# Desired column order
# ---------------------------
desired_col_order = [

    "DCL Manifest",
    "DCL Premanifest",

    *[f"Cluster {c}" for c in groupings['cluster']],

    *[f"HDISS {c}" for c in groupings['hdiss_stage_imp']],

    *[f"Shoulson {s}" for s in groupings['shoulson_stage']]
]

desired_col_order = [
    col for col in desired_col_order
    if col in combined_df.columns
]

combined_df = combined_df[desired_col_order]

# ---------------------------
# Display result
# ---------------------------
print(combined_df)

# Optional export
# combined_df.to_csv("median_iqr_features_all_stages_wide.csv")

combined_df

In [ ]:
'MEAN + [95% CI] for actual features'

import pandas as pd
import numpy as np

# Display settings (so all columns appear in console)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Ensure numeric types
features = kept_features
merged_df[features] = merged_df[features].apply(pd.to_numeric, errors='coerce')
merged_df['tfcscore'] = pd.to_numeric(merged_df['tfcscore'], errors='coerce')
merged_df['diagconf'] = pd.to_numeric(merged_df['diagconf'], errors='coerce')
merged_df['hdiss_stage_imp'] = pd.to_numeric(merged_df['hdiss_stage_imp'], errors='coerce')

# ---------------------------
# Calculate Shoulson Stage
# ---------------------------
def assign_shoulson_stage(tfc):
    if tfc >= 11:
        return 'Stage I'
    elif 7 <= tfc <= 10:
        return 'Stage II'
    elif 4 <= tfc <= 6:
        return 'Stage III'
    elif 1 <= tfc <= 3:
        return 'Stage IV'
    elif tfc == 0:
        return 'Stage V'
    else:
        return np.nan

merged_df['shoulson_stage'] = merged_df['tfcscore'].apply(assign_shoulson_stage)

# ---------------------------
# DCL-based stage
# ---------------------------
merged_df['manifest_status'] = np.where(
    merged_df['diagconf'] == 4,
    'Manifest',
    'Premanifest'
)

# ---------------------------
# Helper function: Median ± IQR
# ---------------------------
def mean_ci_str(vals):
    if len(vals) == 0:
        return np.nan

    mean = vals.mean()
    std = vals.std(ddof=1)
    n = len(vals)

    se = std / np.sqrt(n)
    ci = 1.96 * se

    lower = mean - ci
    upper = mean + ci

    return f"{mean:.2f} [{lower:.2f}–{upper:.2f}]"

# ---------------------------
# Define grouping systems
# ---------------------------
groupings = {
    'manifest_status': ['Manifest', 'Premanifest'],
    'cluster': sorted(merged_df['cluster'].dropna().unique()),
    'hdiss_stage_imp': sorted(merged_df['hdiss_stage_imp'].dropna().unique()),
    'shoulson_stage': ['Stage I','Stage II','Stage III','Stage IV','Stage V']
}

# ---------------------------
# Compute tables for each staging system
# ---------------------------
dfs = {}

for group_name, categories in groupings.items():

    rows = []

    for feat in features:

        row = {'feature': feat}

        for cat in categories:

            vals = merged_df.loc[
                merged_df[group_name] == cat, feat
            ].dropna()

            row[cat] = mean_ci_str(vals)

        rows.append(row)

    dfs[group_name] = pd.DataFrame(rows).set_index('feature')

# ---------------------------
# Rename columns clearly
# ---------------------------
dfs['manifest_status'].columns = [f"DCL {c}" for c in dfs['manifest_status'].columns]
dfs['cluster'].columns = [f"Cluster {c}" for c in dfs['cluster'].columns]
dfs['hdiss_stage_imp'].columns = [f"HDISS {c}" for c in dfs['hdiss_stage_imp'].columns]
dfs['shoulson_stage'].columns = [f"Shoulson {c}" for c in dfs['shoulson_stage'].columns]

# ---------------------------
# Combine all staging systems
# ---------------------------
combined_df = pd.concat(dfs.values(), axis=1)

# ---------------------------
# Desired column order
# ---------------------------
desired_col_order = [

    "DCL Manifest",
    "DCL Premanifest",

    *[f"Cluster {c}" for c in groupings['cluster']],

    *[f"HDISS {c}" for c in groupings['hdiss_stage_imp']],

    *[f"Shoulson {s}" for s in groupings['shoulson_stage']]
]

desired_col_order = [
    col for col in desired_col_order
    if col in combined_df.columns
]

combined_df = combined_df[desired_col_order]

# ---------------------------
# Display result
# ---------------------------
print(combined_df)

# Optional export
# combined_df.to_csv("median_iqr_features_all_stages_wide.csv")

combined_df

In [ ]:
'MEAN + [95% CI] for the standarized features'

import pandas as pd
import numpy as np

# Display settings (so all columns appear in console)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Ensure numeric types
features = kept_features
merged_df_z[features] = merged_df_z[features].apply(pd.to_numeric, errors='coerce')
merged_df_z['tfcscore'] = pd.to_numeric(merged_df_z['tfcscore'], errors='coerce')
merged_df_z['diagconf'] = pd.to_numeric(merged_df_z['diagconf'], errors='coerce')
merged_df_z['hdiss_stage_imp'] = pd.to_numeric(merged_df_z['hdiss_stage_imp'], errors='coerce')

# ---------------------------
# Calculate Shoulson Stage
# ---------------------------
def assign_shoulson_stage(tfc):
    if tfc >= 11:
        return 'Stage I'
    elif 7 <= tfc <= 10:
        return 'Stage II'
    elif 4 <= tfc <= 6:
        return 'Stage III'
    elif 1 <= tfc <= 3:
        return 'Stage IV'
    elif tfc == 0:
        return 'Stage V'
    else:
        return np.nan

merged_df_z['shoulson_stage'] = merged_df_z['tfcscore'].apply(assign_shoulson_stage)

# ---------------------------
# DCL-based stage
# ---------------------------
merged_df_z['manifest_status'] = np.where(
    merged_df_z['diagconf'] == 4,
    'Manifest',
    'Premanifest'
)

# ---------------------------
# Helper function: Median ± IQR
# ---------------------------
def mean_ci_str(vals):
    if len(vals) == 0:
        return np.nan

    mean = vals.mean()
    std = vals.std(ddof=1)
    n = len(vals)

    se = std / np.sqrt(n)
    ci = 1.96 * se

    lower = mean - ci
    upper = mean + ci

    return f"{mean:.2f} [{lower:.2f}–{upper:.2f}]"

# ---------------------------
# Define grouping systems
# ---------------------------
groupings = {
    'manifest_status': ['Manifest', 'Premanifest'],
    'cluster': sorted(merged_df_z['cluster'].dropna().unique()),
    'hdiss_stage_imp': sorted(merged_df_z['hdiss_stage_imp'].dropna().unique()),
    'shoulson_stage': ['Stage I','Stage II','Stage III','Stage IV','Stage V']
}

# ---------------------------
# Compute tables for each staging system
# ---------------------------
dfs = {}

for group_name, categories in groupings.items():

    rows = []

    for feat in features:

        row = {'feature': feat}

        for cat in categories:

            vals = merged_df_z.loc[
                merged_df[group_name] == cat, feat
            ].dropna()

            row[cat] = mean_ci_str(vals)

        rows.append(row)

    dfs[group_name] = pd.DataFrame(rows).set_index('feature')

# ---------------------------
# Rename columns clearly
# ---------------------------
dfs['manifest_status'].columns = [f"DCL {c}" for c in dfs['manifest_status'].columns]
dfs['cluster'].columns = [f"Cluster {c}" for c in dfs['cluster'].columns]
dfs['hdiss_stage_imp'].columns = [f"HDISS {c}" for c in dfs['hdiss_stage_imp'].columns]
dfs['shoulson_stage'].columns = [f"Shoulson {c}" for c in dfs['shoulson_stage'].columns]

# ---------------------------
# Combine all staging systems
# ---------------------------
combined_df = pd.concat(dfs.values(), axis=1)

# ---------------------------
# Desired column order
# ---------------------------
desired_col_order = [

    "DCL Manifest",
    "DCL Premanifest",

    *[f"Cluster {c}" for c in groupings['cluster']],

    *[f"HDISS {c}" for c in groupings['hdiss_stage_imp']],

    *[f"Shoulson {s}" for s in groupings['shoulson_stage']]
]

desired_col_order = [
    col for col in desired_col_order
    if col in combined_df.columns
]

combined_df = combined_df[desired_col_order]

# ---------------------------
# Display result
# ---------------------------
print(combined_df)

# Optional export
# combined_df.to_csv("median_iqr_features_all_stages_wide.csv")

combined_df

In [ ]:
'''
import pandas as pd
import numpy as np

# ================================
# 1️⃣ Add stage labels
# ================================
feature_df["dcl_stage"] = np.where(feature_df["diagconf"] < 4, "Premanifest", "Manifest")
feature_df["cluster_stage"] = feature_df["cluster"].astype(str)

# ================================
# 2️⃣ Define grouping combinations
# ================================
groupings = {
    "DCL Stage": "dcl_stage",
    "Shoulson Stage": "shoulson_stage",
    "Cluster Stage": "cluster_stage"
}

# ================================
# 3️⃣ Function to summarize features
# ================================
def summarize_features(df, group_col, features):
    summary = []
    for grp, grp_df in df.groupby(group_col):
        for feat in features:
            values = grp_df[feat].dropna()
            summary.append({
                "Feature": feat,
                "Group": grp,
                "N": len(values),
                "Mean": values.mean(),
                "Min": values.min(),
                "Max": values.max()
            })
    return pd.DataFrame(summary)

# ================================
# 4️⃣ Generate tables for each grouping
# ================================
tables = {}
for name, col in groupings.items():
    tables[name] = summarize_features(feature_df, col, cluster_features)

# ================================
# 5️⃣ Display / export
# ================================
for name, table in tables.items():
    print(f"\n=== Feature Distribution by {name} ===\n")
    print(table.head(100))  # show first 20 rows for brevity
    # Optional: export to CSV
    # table.to_csv(f"feature_distribution_by_{name.replace(' ','_')}.csv", index=False)
'''

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Choose the grouping: 'dcl_stage', 'shoulson_stage', or 'cluster_stage'
group_col = "cluster"  # can switch to "dcl_stage" or "shoulson_stage"

cols = [feat for feat in cluster_features if feat not in posthoc_features]
# Compute mean for each feature per group
mean_df = feature_df_z.groupby(group_col)[cols].mean().reset_index()

# Melt the dataframe to long format for seaborn
plot_df = mean_df.melt(id_vars=group_col, var_name="Feature", value_name="MeanValue")
plt.figure(figsize=(18,6))
sns.lineplot(
    data=plot_df,
    x="Feature",
    y="MeanValue",
    hue=group_col,
    marker="o",
    palette="husl"
)
plt.xticks(rotation=90)
plt.ylabel("Mean Standardized Value")
plt.title("Feature Trajectories Across Discovered Clusters")
plt.legend(title=group_col, bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1️⃣ Define Shoulson stages
# ------------------------------------------------------------
def shoulson_stage(tfc):
    if tfc >= 11:
        return "Stage I"
    elif tfc >= 7:
        return "Stage II"
    elif tfc >= 3:
        return "Stage III"
    elif tfc >= 1:
        return "Stage IV"
    else:
        return "Stage V"
'''
feature_df["shoulson_stage"] = feature_df_z["tfcscore"].apply(shoulson_stage)
'''
cols = [feat for feat in cluster_features if feat not in posthoc_features]

# ------------------------------------------------------------
# 2️⃣ Include all staging systems
# ------------------------------------------------------------
staging_systems = {
    "Discovered Stages": "cluster",
    "DCL-based Stages": "manifest_status",
    "Shoulson Stages": "shoulson_stage",
    "HD-ISS Stages": "hdiss_stage_imp"
}

# ------------------------------------------------------------
# 3️⃣ Define consistent color palette across all stages
# ------------------------------------------------------------
# Flatten all stage values across all staging systems
all_stages = pd.unique(feature_df_z[list(staging_systems.values())].values.ravel())
all_stages = [str(s) for s in all_stages if pd.notnull(s)]  # remove NaNs & convert to string
all_stages_sorted = sorted(all_stages)
palette = dict(zip(all_stages_sorted, sns.color_palette("husl", n_colors=len(all_stages_sorted))))

# ------------------------------------------------------------
# 4️⃣ Plot feature trajectories with 95% CI shading
# ------------------------------------------------------------
fig, axes = plt.subplots(len(staging_systems), 1, figsize=(20, 20), sharex=True)

for ax, (system_name, col) in zip(axes, staging_systems.items()):

    stages = sorted(feature_df_z[col].dropna().unique(), key=lambda x: str(x))

    # create palette specifically for this staging system
    colors = sns.color_palette("husl", n_colors=len(stages))
    palette = dict(zip(stages, colors))

    for stage in stages:

        subset = feature_df_z[feature_df_z[col] == stage][cols]

        if subset.empty:
            continue

        mean_vals = subset.mean()
        sem_vals = subset.std(ddof=1) / np.sqrt(len(subset))
        ci95 = 1.96 * sem_vals

        x = np.arange(len(cols))

        # plot mean
        ax.plot(x, mean_vals, marker='o',
                color=palette[stage],
                label=str(stage))

        # CI shading
        ax.fill_between(
            x,
            mean_vals - ci95,
            mean_vals + ci95,
            color=palette[stage],
            alpha=0.15
        )

    ax.set_title(f"Mean feature profiles across stages — {system_name}", fontsize=18)
    ax.set_ylabel("Mean Standardized Value")
    ax.set_xticks(np.arange(len(cols)))
    ax.set_xticklabels(cols, rotation=90)
    ax.legend(title=system_name, bbox_to_anchor=(1.0, 1), loc="upper left")

plt.xlabel("Clinical Features")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(len(staging_systems), 1, figsize=(20, 20), sharex=True)

for ax, (system_name, col) in zip(axes, staging_systems.items()):

    stages = sorted(feature_df_z[col].dropna().unique(), key=lambda x: str(x))

    colors = sns.color_palette("husl", n_colors=len(stages))
    palette = dict(zip(stages, colors))

    for stage in stages:

        subset = feature_df_z[feature_df_z[col] == stage][cols]

        if subset.empty:
            continue

        # median and IQR
        median_vals = subset.median()
        lower_percentile = subset.quantile(0.25)
        upper_percentile = subset.quantile(0.75)

        x = np.arange(len(cols))

        # median line
        ax.plot(
            x,
            median_vals,
            marker='o',
            color=palette[stage],
            label=str(stage)
        )

        # IQR shading
        ax.fill_between(
            x,
            lower_percentile,
            upper_percentile,
            color=palette[stage],
            alpha=0.15
        )

    ax.set_title(f"Feature profiles across stages — {system_name}", fontsize=18)
    ax.set_ylabel("Feature Value (Median with IQR)")
    ax.set_xticks(np.arange(len(cols)))
    ax.set_xticklabels(cols, rotation=90)
    ax.legend(title=system_name, bbox_to_anchor=(1.0, 1), loc="upper left")

plt.xlabel("Clinical Features")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Prepare data
data = feature_df_z[['age','cluster','tfcscore']].dropna()
data['age'] = data['age'].astype(int)

# Convert TFC score to percentage (max TFC = 13)
data['tfc_percent'] = (data['tfcscore'] / 13) * 100

# Plot bell-shaped KDE curves
plt.figure(figsize=(10,6))

ax = sns.kdeplot(
    data=data,
    x='age',
    weights='tfc_percent',
    hue='cluster',
    fill=True,
    common_norm=False,
    alpha=0.4
)

plt.xlabel("Age")
plt.ylabel("TFC Score Density (%)")
plt.title("Bell-Shaped Distribution of TFC Score Across Age by Discovered Stage")

# Add legend
plt.legend(title="Cluster (Stage)")

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Prepare data
data = feature_df_z[['age','cluster','motscore']].dropna()
data['age'] = data['age'].astype(int)

# Convert TFC score to percentage (max TFC = 13)
data['mot_percent'] = (data['motscore'] / 13) * 100

# Plot bell-shaped KDE curves
plt.figure(figsize=(10,6))

sns.kdeplot(
    data=data,
    x='age',
    weights='mot_percent',   # weight by TFC percentage
    hue='cluster',           # one curve per stage
    fill=True,
    common_norm=False,
    alpha=0.4
)

plt.xlabel("Age")
plt.ylabel("Motor Score Density (%)")
plt.title("Bell-Shaped Distribution of Motor Score Across Age by Discovered Stage")

plt.legend(title="Cluster (Stage)")
plt.tight_layout()
plt.show()


In [ ]:
"""
import pandas as pd
import numpy as np

# Mapping grouping column names to friendly staging system names & ordered stages
staging_order = {
    'dcl_stage': ['manifest', 'premanifest'],
    'cluster': sorted(merged_df['cluster'].dropna().unique()),
    'hdiss_stage_imp': sorted(merged_df['hdiss_stage_imp'].dropna().unique()),
    'shoulson_stage': ['Stage 1', 'Stage 2', 'Stage 3', 'Stage 4', 'Stage 5']
}

staging_names = {
    'dcl_stage': 'DCL-based Stages',
    'cluster': 'Discovered Stages',
    'hdiss_stage_imp': 'HDISS Stages',
    'shoulson_stage': 'Shoulson Stages'
}

# Format mean ± ci string
def format_mean_ci(row):
    mean = row['mean']
    ci_low = row['ci_low']
    ci_high = row['ci_high']
    # Use ± half CI width; handle NaNs gracefully
    half_ci = np.nan if pd.isna(ci_low) or pd.isna(ci_high) else (ci_high - ci_low) / 2
    if pd.isna(mean) or pd.isna(half_ci):
        return "nan"
    return f"{mean:.2f} ± {half_ci:.2f}"

wide_tables = []

for grp in staging_order.keys():
    # Filter summary_df for this grouping
    df_group = summary_df[summary_df[grp].notnull()].copy()

    # Format the mean ± ci string column
    df_group['mean_ci'] = df_group.apply(format_mean_ci, axis=1)

    # Pivot to wide format: index=feature, columns=stage, values=mean_ci string
    df_pivot = df_group.pivot(index='feature', columns=grp, values='mean_ci')

    # Reorder columns according to known order
    order = staging_order[grp]
    # If any stage missing in data, ignore it
    cols = [col for col in order if col in df_pivot.columns]
    df_pivot = df_pivot[cols]

    # Rename columns to add staging system name prefix for clarity
    df_pivot.columns = pd.MultiIndex.from_product([[staging_names[grp]], df_pivot.columns])

    wide_tables.append(df_pivot)

# Combine all staging systems horizontally (along columns)
final_table = pd.concat(wide_tables, axis=1)

# Optionally sort rows alphabetically by feature
final_table = final_table.sort_index()

# Display
print(final_table.head(20))

# To save the table to Excel (preserves multi-index columns)
# final_table.to_excel("combined_staging_feature_summary.xlsx")
final_table
"""

In [ ]:
a=feature_df['hdiss_stage_imp'].value_counts()
print(a)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import StandardScaler
import numpy as np

# ----------------------------
# 1️⃣ Define Feature Groups
# ----------------------------
motor_features = [
    "ocularh","ocularv","sacinith","sacinitv","sacvelh","sacvelv",
    "dysarth","tongue","fingtapr","fingtapl","prosupr","prosupl",
    "luria","rigarmr","rigarml","brady",
    "dysttrnk","dystrue","dystlue","dystrle","dystlle",
    "chorface","chorbol","chortrnk","chorrue","chorlue","chorrle","chorlle",
    "gait","tandem","retropls"
]

cognitive_features = ["mmsetotal","sit1","verflt05","swrt1","scnt1","verfct5","sdmt1"]

functional_features = ["indepscl","carelevl","adl","chores","finances","occupatn"]

feature_groups = {
    "Motor Measures": motor_features,
    "Cognitive Measures": cognitive_features,
    "Functional Measures": functional_features
}

feature_df["cluster"] = feature_df["cluster"].astype(str)

# ----------------------------
# 2️⃣ Loop per Domain
# ----------------------------
i=0

for title, features in feature_groups.items():

    features_present = [f for f in features if f in feature_df.columns]
    if not features_present:
        continue

    df_domain = feature_df[["cluster"] + features_present].dropna()

    # Standardize features
    scaler = StandardScaler()
    scaled_vals = scaler.fit_transform(df_domain[features_present])
    scaled_df = pd.DataFrame(scaled_vals, columns=features_present, index=df_domain.index)

    # Reverse direction where lower = worse
    if title == "Cognitive Measures":
        scaled_df *= -1
    if title == "Functional Measures":
        scaled_df *= -1

    # Create domain severity index
    df_domain["severity_index"] = scaled_df.mean(axis=1)

    # Order clusters mild → severe
    cluster_order = (
        df_domain.groupby("cluster")["severity_index"]
        .mean()
        .sort_values()
        .index
        .tolist()
    )

    feature_df["cluster"] = pd.Categorical(
        feature_df["cluster"],
        categories=cluster_order,
        ordered=True
    )

    print(f"{title} cluster order (mild → severe):", cluster_order)

    # ----------------------------
    # 3️⃣ Compute Means for Plot
    # ----------------------------
    cluster_means = (
        feature_df.groupby("cluster")[features_present]
        .mean()
        .reset_index()
    )

    df_melt = cluster_means.melt(
        id_vars="cluster",
        value_vars=features_present,
        var_name="Feature",
        value_name="Mean Value"
    )

    palette = sns.color_palette("husl", len(features_present))
    feature_colors = dict(zip(features_present, palette))

    plt.figure(figsize=(20,6))
    sns.barplot(
        x="cluster",
        y="Mean Value",
        hue="Feature",
        data=df_melt,
        palette=feature_colors
    )

    plt.title(title, fontsize=16)
    plt.xlabel("Cluster (Mild → Severe within Domain)", fontsize=16)
    plt.ylabel("Mean Value", fontsize=16)
    if i==0:
      plt.legend(
        title="Feature",
        loc="upper center",
        bbox_to_anchor=(0.5, -0.7),
        ncol=6,
        frameon=False
       )
    else:
      plt.legend(
          title="Feature",
          loc="upper center",
          bbox_to_anchor=(0.5, -0.25),
          ncol=6,
          frameon=False
         )

    plt.tight_layout()
    plt.show()
    i+=1


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import pandas as pd

sns.set(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (14,6)

# Define subdomains
motor_subdomains = {
    "Oculomotor": ["ocularh","ocularv","sacinith","sacinitv","sacvelh","sacvelv"],
    "Speech/Upper Limb": ["dysarth","tongue","fingtapr","fingtapl","prosupr","prosupl",
                          "luria","rigarmr","rigarml","brady"],
    "Trunk/Lower Limb": ["dysttrnk","dystrue","dystlue","dystrle","dystlle",
                         "chorface","chorbol","chortrnk","chorrue","chorlue","chorrle","chorlle",
                         "gait","tandem","retropls"]
}

# Filter features present in the dataframe
for k in motor_subdomains:
    motor_subdomains[k] = [f for f in motor_subdomains[k] if f in feature_df.columns]

# Z-score features
feature_df_z = feature_df.copy()
scaler = StandardScaler()
all_motor_features = sum(motor_subdomains.values(), [])
feature_df_z[all_motor_features] = scaler.fit_transform(feature_df_z[all_motor_features])

# Compute subdomain average per age
subdomain_age = pd.DataFrame({"age": feature_df_z["age"].unique()})
for subdomain, features in motor_subdomains.items():
    subdomain_age[subdomain] = feature_df_z.groupby("age")[features].mean().values

# Plot
plt.figure(figsize=(14,6))
for subdomain in motor_subdomains.keys():
    plt.plot(subdomain_age["age"], subdomain_age[subdomain], label=subdomain, linewidth=3)

plt.xlabel("Age")
plt.ylabel("Z-scored Motor Severity")
plt.title("Motor Subdomain Progression Across Age")
plt.legend(title="Motor Subdomains")
plt.show()


In [ ]:
from scipy.stats import kruskal
import pandas as pd

# Features to test
features = variables  # or extend with other features

kruskal_results = {}

for feature in features:
    # Drop NaNs per group
    groups = [group[feature].dropna().values for name, group in df_plot.groupby("cluster")]
    H, p = kruskal(*groups)
    kruskal_results[feature] = {"H-statistic": H, "p-value": p}

kruskal_df = pd.DataFrame(kruskal_results).T
kruskal_df

!pip install scikit_posthocs
import scikit_posthocs as sp

dunn_results = {}

for feature in features:
    # Drop rows with NaN in the current feature
    df_temp = df_plot[["cluster", feature]].dropna()
    # Dunn's test
    dunn = sp.posthoc_dunn(df_temp, val_col=feature, group_col="cluster", p_adjust="bonferroni")
    dunn_results[feature] = dunn

# Example: view post-hoc table for "mmsetotal"
dunn_results["mmsetotal"]


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

# =================================================
# 0. HARD SAFETY FIX (THIS PREVENTS THE ERROR)
# =================================================
feature_df = feature_df.copy()
feature_df["cluster"] = feature_df["cluster"].astype(str)
feature_df["age"] = feature_df["age"].astype(float)

# =================================================
# 1. Define Shoulson stage
# =================================================
def shoulson_stage(tfc):
    if tfc >= 11:
        return "Stage I"
    elif tfc >= 7:
        return "Stage II"
    elif tfc >= 3:
        return "Stage III"
    else:
        return "Stage IV–V"

feature_df["shoulson"] = feature_df["tfcscore"].apply(shoulson_stage)

# =================================================
# 2. Age-wise percentage distributions (NO BINNING)
# =================================================
dist_cluster = (
    feature_df
    .groupby(["age", "cluster"])
    .size()
    .groupby(level=0)
    .apply(lambda x: 100 * x / x.sum())
    .unstack(fill_value=0)
    .sort_index()
)

dist_shoulson = (
    feature_df
    .groupby(["age", "shoulson"])
    .size()
    .groupby(level=0)
    .apply(lambda x: 100 * x / x.sum())
    .unstack(fill_value=0)
    .sort_index()
)

# =================================================
# 3. Ordered categories (SAFE)
# =================================================
cluster_order = sorted(dist_cluster.columns, key=lambda x: int(x))
shoulson_order = ["Stage I", "Stage II", "Stage III", "Stage IV–V"]

# =================================================
# 4. Fancy color palettes
# =================================================
cluster_palette = sns.color_palette(
    "blend:#A9CCE3,#1F618D", n_colors=len(cluster_order)
)
shoulson_palette = sns.color_palette(
    "blend:#FAD7A0,#922B21", n_colors=len(shoulson_order)
)

cluster_colors = dict(zip(cluster_order, cluster_palette))
shoulson_colors = dict(zip(shoulson_order, shoulson_palette))

# =================================================
# 5. Plot stacked bars + percentages
# =================================================
ages = dist_cluster.index.get_level_values(0).values # Fixed: Extract single age values
x = np.arange(len(ages))
width = 0.38

fig, ax = plt.subplots(figsize=(18, 6))

# -------- Discovered clusters --------
bottom = np.zeros(len(ages))
for c in cluster_order:
    ax.bar(
        x - width/2,
        dist_cluster[c],
        width,
        bottom=bottom,
        color=cluster_colors[c],
        edgecolor="white",
        linewidth=0.7
    )

    for i, val in enumerate(dist_cluster[c].values):
        if val >= 6:
            ax.text(
                x[i] - width/2,
                bottom[i] + val/2,
                f"{val:.0f}%",
                ha="center",
                va="center",
                fontsize=9,
                color="white",
                fontweight="bold"
            )

    bottom += dist_cluster[c].values

# -------- Shoulson stage --------
bottom = np.zeros(len(ages))
for s in shoulson_order:
    ax.bar(
        x + width/2,
        dist_shoulson[s],
        width,
        bottom=bottom,
        color=shoulson_colors[s],
        edgecolor="white",
        linewidth=0.7
    )

    for i, val in enumerate(dist_shoulson[s].values):
        if val >= 6:
            ax.text(
                x[i] + width/2,
                bottom[i] + val/2,
                f"{val:.0f}%",
                ha="center",
                va="center",
                fontsize=9,
                color="white",
                fontweight="bold"
            )

    bottom += dist_shoulson[s].values

# =================================================
# 6. Axes & style
# =================================================
ax.set_xticks(x)
ax.set_xticklabels(ages.astype(int), rotation=45)
ax.set_xlabel("Age", fontsize=12)
ax.set_ylabel("Percentage of patients (%)", fontsize=12)
ax.set_ylim(0, 100)

ax.set_title(
    "Age-wise percentage distribution of Discovered Clusters vs Shoulson Stage",
    fontsize=15,
    fontweight="bold"
)

sns.despine(ax=ax)

# =================================================
# 7. Legends
# =================================================
cluster_legend = [
    Patch(facecolor=cluster_colors[c], label=f"Cluster {c}")
    for c in cluster_order
]

shoulson_legend = [
    Patch(facecolor=shoulson_colors[s], label=s)
    for s in shoulson_order
]

leg1 = ax.legend(
    handles=cluster_legend,
    title="Discovered clusters",
    bbox_to_anchor=(1.02, 1.0),
    loc="upper left",
    frameon=False
)
ax.add_artist(leg1)

ax.legend(
    handles=shoulson_legend,
    title="Shoulson stage",
    bbox_to_anchor=(1.02, 0.55),
    loc="upper left",
    frameon=False
)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# Example: feature_df must have columns 'subjid', 'age', 'cluster'
# feature_df = pd.read_csv("your_data.csv")  # already exists

# Sort by subject and age
feature_df_sorted = feature_df.sort_values(["subjid", "age"])

# Ensure 'age' is numeric and 'cluster' is string for consistency
feature_df_sorted['age'] = pd.to_numeric(feature_df_sorted['age'], errors='coerce')
feature_df_sorted.dropna(subset=['age'], inplace=True)
feature_df_sorted['cluster'] = feature_df_sorted['cluster'].astype(str)

# Build unique node labels for cluster at each age
all_ages = sorted([int(a) for a in feature_df_sorted.age.unique()]) # Explicitly convert each unique age to int
all_clusters = sorted(feature_df_sorted.cluster.unique()) # These are already strings
node_labels = [f"C{c} (Age {a})" for a in all_ages for c in all_clusters]

# Mapping for node indices
node_index_map = {f"C{c}_A{a}": i for i, (a, c) in enumerate(
    [(a, c) for a in all_ages for c in all_clusters])}

# Collect links: source → target per patient trajectory
links = []
patient_colors = {}  # assign a color per unique trajectory

# Generate a color palette
import plotly.express as px
palette = px.colors.qualitative.Plotly
color_idx = 0

# Group by patient to follow transitions
for subjid, group in feature_df_sorted.groupby("subjid"):
    # Convert ages and clusters in traj to integer/string for key consistency
    traj = list(zip(group.age.astype(int), group.cluster.astype(str)))
    # Convert to consecutive node indices
    for i in range(len(traj)-1):
        src = node_index_map[f"C{traj[i][1]}_A{traj[i][0]}"]
        tgt = node_index_map[f"C{traj[i+1][1]}_A{traj[i+1][0]}"]

        # Use the same color for this patient's trajectory
        if subjid not in patient_colors:
            patient_colors[subjid] = palette[color_idx % len(palette)]
            color_idx += 1
        links.append((src, tgt, patient_colors[subjid]))

# Aggregate counts per source-target pair
link_df = pd.DataFrame(links, columns=["source","target","color"])
link_agg = link_df.groupby(["source","target","color"]).size().reset_index(name="value")

# Build Sankey
fig = go.Figure(go.Sankey(
    node=dict(
        label=node_labels,
        pad=15,
        thickness=20,
        color="lightblue"
    ),
    link=dict(
        source=link_agg.source,
        target=link_agg.target,
        value=link_agg.value,
        color=link_agg.color
    )
))

fig.update_layout(title_text="Patient Cluster Transitions Over Age", font_size=12)
fig.show()
